# 0.Config

ส่วนนี้ตรวจสอบเวอร์ชัน Python และแพ็กเกจที่เกี่ยวข้อง 
- เตรียม API key
- กำหนด model ID ที่จะใช้ตลอดการทดลอง เพื่อให้การรัน benchmark ใช้สภาพแวดล้อมที่ระบุได้ชัดเจน

In [ ]:
import asyncio
import hashlib
import json
import math
import os
import re
import sys
import time
import warnings
from collections import Counter, defaultdict
from contextlib import asynccontextmanager, suppress
from copy import deepcopy
from dataclasses import asdict, is_dataclass
from datetime import datetime, timezone
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from typing import Literal
from urllib.parse import urlsplit
from uuid import uuid4

from dotenv import load_dotenv
from IPython.display import Markdown, display
from openai import AsyncOpenAI
from agents import (Agent, Runner, RunConfig, RunHooks, RunContextWrapper,
                    ModelSettings, OpenAIResponsesModel, SQLiteSession, function_tool,
                    WebSearchTool, set_tracing_disabled)
from agents.model_settings import ModelRetrySettings

assert sys.version_info >= (3, 11), "ใช้ Python 3.11 ขึ้นไป (Docker ใช้ 3.12)"
PACKAGE_VERSIONS = {p: version(p) for p in
    ["openai-agents", "openai", "mcp", "fastmcp", "python-dotenv"]}
print("Python:", sys.version.split()[0])
print(json.dumps(PACKAGE_VERSIONS, indent=2))


In [ ]:
# Docker mount ชี้ shared ของ repository root อยู่แล้ว ส่วน local หา root จาก compose
# ไม่สร้าง openai_agent/shared ขึ้นเอง และไม่อาศัย output_path จากหมวดอื่น
repo_root = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "docker-compose.yml").is_file()
                  and (p / "shared" / "benchmark_cases.json").is_file()), None)
if repo_root is not None:
    SHARED_DIR = repo_root / "shared"
    load_dotenv(repo_root / ".env", override=False)
else:
    SHARED_DIR = Path("/workspace/shared")
if not (SHARED_DIR / "benchmark_cases.json").is_file():
    raise FileNotFoundError("ไม่พบ shared ของ repository; ตรวจ Docker volume mapping")
RESULTS_DIR = SHARED_DIR / "results" / "openai_agents"
STATE_DIR = SHARED_DIR / "state" / "openai_agents"
SUMMARY_DIR = SHARED_DIR / "summaries"
for directory in (RESULTS_DIR, STATE_DIR, SUMMARY_DIR):
    directory.mkdir(parents=True, exist_ok=True)
SHARED_CASES = json.loads((SHARED_DIR / "benchmark_cases.json").read_text(encoding="utf-8"))
SHARED_SCORING_SHA256 = hashlib.sha256((SHARED_DIR / "scoring.py").read_bytes()).hexdigest()
# shared cases เป็นคนละชุดกับ 17 เคสนี้ จึงอ่านไว้เพื่อ audit และไม่ใช้แทนโจทย์ต้นฉบับ
print("Shared:", SHARED_DIR)
print("Shared scoring bytes:", (SHARED_DIR / "scoring.py").stat().st_size)

if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter OPENAI_API_KEY: ").strip()
if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("ยังไม่มี OPENAI_API_KEY")
MODEL_INPUT = os.getenv("BENCHMARK_MODEL", "gpt-6-luna").strip()
# openai: เป็น provider prefix ของ LangChain ไม่ใช่ model ID ของ Responses API
MODEL = MODEL_INPUT.removeprefix("openai:")
if not MODEL:
    raise ValueError("กรุณาระบุ BENCHMARK_MODEL")
client = AsyncOpenAI(max_retries=0, timeout=60)
model = OpenAIResponsesModel(model=MODEL, openai_client=client)
BASE_SETTINGS = ModelSettings(preserve_raw_usage=True, retry=ModelRetrySettings(max_retries=0))
set_tracing_disabled(True)  # เก็บ raw responses/hooks ลง JSON ในเครื่อง ไม่ส่ง trace dashboard
RUN_CONFIG = RunConfig(tracing_disabled=True)
print("API key loaded:", bool(os.getenv("OPENAI_API_KEY")))
print("Model:", MODEL)


### Config: ขอบเขตเวลาและข้อจำกัด SDK

คง model ID `gpt-6-luna` และ provider defaults เช่น temperature/reasoning ตามต้นฉบับ; ไม่บังคับ structured output เพิ่มเพื่อช่วยคะแนน ใช้ Responses API, API retry = 0, runner retry = 0; API timeout 60 วินาที (Web 90 วินาที)

`recursion_limit` เดิมนับขั้นตอนของ LangGraph; `max_turns` ใหม่นับรอบเรียกโมเดลใน agent loop และไม่จำกัดจำนวน hosted web actions ภายใน request เดียว จึง **ไม่ใช่ค่าเทียบเท่ากัน** แต่ละ run บันทึกทั้งสองค่าแยกกัน กำหนดงบ OpenAI ล่วงหน้า: Smoke/งานไม่มี tool/Web = 10, Math/MCP = 20 (Math02 = 40), Planning = 40, Error = 10 และไม่เพิ่มงบเพื่อไล่ให้ผ่าน

เวลา agent วัดเฉพาะ `Runner.run` รวม model/tool/session I/O; การเชื่อม MCP และตรวจ state ก่อน restart แยกเป็น setup; ไม่รวมเวลาคนตรวจหรือรอ restart ประวัติที่ถูกส่งซ้ำยังเป็น input tokens ที่ API คิดจริง แต่ไม่นำ usage ของ response เก่ามาบวกซ้ำ

In [ ]:
def jsonable(value):
    if hasattr(value, "model_dump"):
        return value.model_dump(mode="json")
    if is_dataclass(value):
        return {k: jsonable(v) for k, v in asdict(value).items()}
    if isinstance(value, dict):
        return {str(k): jsonable(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(v) for v in value]
    if isinstance(value, Path):
        return str(value)
    return value

def sha256_json(value):
    return hashlib.sha256(json.dumps(jsonable(value), ensure_ascii=False,
        sort_keys=True, allow_nan=False, separators=(",", ":")).encode()).hexdigest()

def parse_json(text):
    try:
        return json.loads(text, parse_constant=lambda value: (_ for _ in ()).throw(ValueError(value)))
    except (TypeError, ValueError):
        return None

def begin_run(case_id, prompt, instructions, *, case_version=None,
              max_turns=10, source_recursion_limit=None, timeout_s=120):
    return {
        "case_id": case_id, "case_version": case_version, "run_id": uuid4().hex,
        "sdk": "openai_agents", "revision": 0, "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "model": MODEL, "model_input": MODEL_INPUT, "versions": deepcopy(PACKAGE_VERSIONS),
        "prompt": prompt, "prompt_sha256": hashlib.sha256(prompt.encode()).hexdigest(),
        "system_prompt": instructions, "runtime_status": "not_started", "error": None,
        "checks": {}, "task_passed": None, "raw_answer": None, "answer": None,
        "tool_trace": [], "raw_responses": [], "run_items": [], "citations": [],
        "usage_per_response": [], "usage": None, "warnings": [], "review": None,
        "tool_call_count": None, "trace_complete": False,
        "latency": {"agent_s": None, "setup_s": 0.0, "cleanup_s": 0.0,
                    "scope": "Runner.run including model, tools and session I/O"},
        "configuration": {"max_turns": max_turns,
            "max_turns_meaning": "model invocations per Runner.run; hosted actions may be internal",
            "source_recursion_limit": source_recursion_limit,
            "source_recursion_meaning": "LangGraph steps; not equivalent to max_turns",
            "timeout_s": timeout_s, "api_timeout_s": 60,
            "api_max_retries": 0, "runner_max_retries": 0,
            "model_settings": BASE_SETTINGS.to_json_dict(),
            "shared_scoring_sha256": SHARED_SCORING_SHA256},
    }

class EvidenceHooks(RunHooks):
    """เก็บเฉพาะ request ใหม่ของ run นี้ รวมหลักฐานบางส่วนเมื่อ run ล้มเหลว"""
    def __init__(self, record):
        self.record = record

    async def on_llm_end(self, context, agent, response):
        index = len(self.record["raw_responses"])
        if isinstance(context.context, dict):
            context.context["write_todos_calls_in_response"] = sum(
                getattr(item, "type", None) == "function_call" and getattr(item, "name", None) == "write_todos"
                for item in response.output)
        self.record["raw_responses"].append(jsonable(response))
        for item in response.output:
            self.record["tool_trace"].append({"kind": "model_output",
                "response_index": index, "raw": jsonable(item)})

    async def on_tool_end(self, context, agent, tool, result):
        self.record["tool_trace"].append({"kind": "tool_result",
            "name": tool.name, "call_id": getattr(context, "tool_call_id", None),
            "output": jsonable(result)})

def calls_of(record):
    calls = []
    for i, event in enumerate(record["tool_trace"]):
        raw = event.get("raw", {})
        if event["kind"] == "model_output" and raw.get("type") == "function_call":
            calls.append({"id": raw.get("call_id"), "name": raw.get("name"),
                "args": parse_json(raw.get("arguments")), "trace_index": i,
                "response_index": event["response_index"]})
    return calls


In [ ]:
def response_usage(record):
    # SDK normalized Usage มี default ศูนย์ จึงใช้ raw_usage ที่ preserve ไว้เท่านั้น
    records = [{"response_id": r.get("response_id"), "usage": r.get("raw_usage")}
               for r in record["raw_responses"]]
    totals = {}
    for key in ("input_tokens", "output_tokens", "total_tokens"):
        values = [r["usage"].get(key) if isinstance(r["usage"], dict) else None for r in records]
        totals[key] = sum(values) if values and all(type(v) is int for v in values) else None
    return records, (totals if records else None)

def refresh_evidence(record):
    record["usage_per_response"], record["usage"] = response_usage(record)
    if not record["trace_complete"]:
        # partial usage เป็น lower bound และไม่อ้างว่า usage รวมครบ
        record["usage_observed_partial"] = record["usage"]
        record["usage"] = None
    hosted = [e["raw"] for e in record["tool_trace"] if e.get("raw", {}).get("type") == "web_search_call"]
    observed = len(calls_of(record)) + len(hosted)
    record["observed_tool_call_count"] = observed
    record["tool_call_count"] = observed if record["trace_complete"] else None
    record["citations"] = [a for r in record["raw_responses"] for item in r.get("output", [])
        if item.get("type") == "message" for block in item.get("content", [])
        for a in block.get("annotations", []) if a.get("type") == "url_citation"]
    record["answer"] = parse_json(record["raw_answer"])
    record["latency_s"] = record["latency"]["agent_s"]

async def run_once(record, agent, *, session=None, context=None):
    """เรียกหนึ่งครั้ง ไม่มี rerun-until-pass; scorer และ fixture อยู่ในหมวดของตน"""
    result = None
    record["runtime_status"] = "running"
    record["configuration"]["model_settings"] = agent.model_settings.to_json_dict()
    started = time.perf_counter()
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        try:
            result = await asyncio.wait_for(
                Runner.run(agent, record["prompt"], session=session, context=context,
                    max_turns=record["configuration"]["max_turns"],
                    hooks=EvidenceHooks(record), run_config=RUN_CONFIG),
                timeout=record["configuration"]["timeout_s"])
            record["raw_answer"] = str(result.final_output).strip() if result.final_output is not None else ""
            record["run_items"] = [{"type": item.type, "raw_item": jsonable(item.raw_item)}
                                   for item in result.new_items]
            record["runtime_status"] = "interrupted" if result.interruptions else "completed"
            record["trace_complete"] = not bool(result.interruptions)
        except Exception as exc:
            record["runtime_status"] = "failed"
            record["error"] = {"type": type(exc).__name__, "message": str(exc),
                "status_code": getattr(exc, "status_code", None),
                "request_id": getattr(exc, "request_id", None)}
        finally:
            record["latency"]["agent_s"] = time.perf_counter() - started
            refresh_evidence(record)
            record["warnings"].extend({"type": w.category.__name__, "message": str(w.message)} for w in caught)
    return result

def final_complete(record):
    return record["runtime_status"] == "completed" and bool(record["raw_answer"])

def score(record, checks):
    # runtime ไม่สำเร็จ: ประเมินคุณภาพคำตอบไม่ได้; เก็บ partial trace/log แยกไว้
    if record["runtime_status"] != "completed":
        record["checks"] = {k: None for k in checks}
        record["task_passed"] = None
    else:
        if any(v is not None and type(v) is not bool for v in checks.values()):
            raise TypeError("checks ต้องเป็น bool หรือ None")
        record["checks"] = checks
        record["task_passed"] = False if any(v is False for v in checks.values()) else (
            True if checks and all(v is True for v in checks.values()) else None)
    return record["task_passed"]

def show_run(record):
    print(record["case_id"], "|", record["runtime_status"], "| run_id:", record["run_id"])
    print("Answer:", record["raw_answer"])
    print("Error:", record["error"])
    print("Checks:", json.dumps(record["checks"], ensure_ascii=False, indent=2))
    print("Task passed:", record["task_passed"], "| Usage:", record["usage"])


In [ ]:
def export_results(category, records):
    payload = {"schema_version": "1.0", "saved_at_utc": datetime.now(timezone.utc).isoformat(),
        "sdk": "openai_agents", "category": category, "results": deepcopy(records)}
    serialized = json.dumps(payload, ensure_ascii=False, indent=2, allow_nan=False)
    path = RESULTS_DIR / f"{category}_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}_{uuid4().hex}.json"
    with path.open("x", encoding="utf-8") as f:
        f.write(serialized)
    if json.loads(path.read_text(encoding="utf-8")) != payload:
        raise IOError("ข้อมูลที่อ่านกลับไม่ตรงกับที่เขียน")
    return path


### Config: Manual review ที่ผูกกับหลักฐาน

Web และ Planning มีเกณฑ์เชิงเนื้อหาที่ตรวจอัตโนมัติไม่ได้ครบ ให้เก็บ JSON review template ที่คะแนนเริ่มเป็น `null` อ่านคำตอบ/raw responses/tool trace ของ run นั้น แล้วแก้เฉพาะคะแนนพร้อมเหตุผลและหลักฐานในไฟล์ review ก่อนเรียก `apply_review` และ export ใหม่

Review ต้องมี `run_id` และ `evidence_sha256` ตรงกันทุกครั้ง การรันใหม่จะไม่รับ review เก่า คะแนนอัตโนมัติที่เป็น `false` ถูก review ทับให้ผ่านไม่ได้

In [ ]:
def evidence_hash(record):
    keys = ("run_id", "case_id", "case_version", "model", "prompt", "system_prompt", "raw_answer",
            "tool_trace", "raw_responses", "server_trace", "turns", "source_ownership")
    return sha256_json({k: record.get(k) for k in keys})

def review_template(record, rubric):
    record["review_rubric"] = rubric
    record["review"] = {"status": "pending", "run_id": record["run_id"],
        "evidence_sha256": evidence_hash(record), "checks": {k: None for k in rubric}}
    return {**deepcopy(record["review"]), "reviewer": "", "notes": "", "evidence": [], "rubric": rubric}

def save_review_template(record):
    directory = STATE_DIR / "reviews" / record["run_id"]
    directory.mkdir(parents=True, exist_ok=True)
    path = directory / f"review_{uuid4().hex}.json"
    template = review_template(record, record["review_rubric"])
    path.write_text(json.dumps(template, ensure_ascii=False, indent=2), encoding="utf-8")
    return path

def apply_review(record, path):
    review = json.loads(Path(path).read_text(encoding="utf-8"))
    if review.get("run_id") != record["run_id"] or review.get("evidence_sha256") != evidence_hash(record):
        raise ValueError("review เป็นของ run หรือหลักฐานคนละชุด")
    if set(review.get("checks", {})) != set(record["review_rubric"]):
        raise ValueError("รายการเกณฑ์ review ไม่ตรงกับ rubric")
    if not review.get("reviewer") or not review.get("notes") or not review.get("evidence"):
        raise ValueError("ระบุผู้ตรวจ เหตุผล และหลักฐานที่อ่านจริงก่อนให้คะแนน")
    if any(v is not None and type(v) is not bool for v in review["checks"].values()):
        raise ValueError("คะแนน review ใช้ true / false / null เท่านั้น")
    previous_hash = sha256_json(record)
    checks = deepcopy(record["automatic_checks"])
    for key, value in review["checks"].items():
        if key in checks and checks[key] is None:
            checks[key] = value
    checks.update({"review:" + k: v for k, v in review["checks"].items()})
    record["supersedes_record_sha256"] = previous_hash
    record["revision"] = record.get("revision", 0) + 1
    record["review"] = deepcopy(review)
    record["review"]["status"] = "reviewed" if all(v is not None for v in review["checks"].values()) else "pending"
    score(record, checks)

def add_review(record, rubric):
    record["automatic_checks"] = deepcopy(record["checks"])
    review_template(record, rubric)
    score(record, {**record["checks"], **{"review:" + k: None for k in rubric}})


In [ ]:
def matches_expected(actual, expected):
    """Compare structure exactly; allow 0.01 tolerance for numbers."""
    if isinstance(expected, dict):
        return (
            isinstance(actual, dict)
            and set(actual) == set(expected)
            and all(
                matches_expected(actual[key], value)
                for key, value in expected.items()
            )
        )

    if isinstance(expected, list):
        return isinstance(actual, list) and actual == expected

    if type(expected) in (int, float):
        return (
            type(actual) in (int, float)
            and math.isfinite(actual)
            and math.isclose(actual, expected, rel_tol=0, abs_tol=0.01)
        )

    return type(actual) is type(expected) and actual == expected


In [ ]:
def combine_turns(record, turns):
    record["turns"] = deepcopy(turns)
    record["prompt"] = [t["prompt"] for t in turns]
    record["prompt_sha256"] = sha256_json(record["prompt"])
    record["raw_responses"] = []
    record["tool_trace"] = []
    record["run_items"] = []
    record["warnings"] = []
    for turn in turns:
        response_offset = len(record["raw_responses"])
        record["raw_responses"].extend(deepcopy(turn["raw_responses"]))
        for event in deepcopy(turn["tool_trace"]):
            if "response_index" in event:
                event["response_index"] += response_offset
            record["tool_trace"].append(event)
        record["run_items"].extend(deepcopy(turn["run_items"]))
        record["warnings"].extend(deepcopy(turn["warnings"]))
    record["raw_answer"] = turns[-1]["raw_answer"] if turns else None
    record["runtime_status"] = "completed" if turns and all(t["runtime_status"] == "completed" for t in turns) else "failed"
    record["error"] = next((t["error"] for t in turns if t["error"]), None)
    record["trace_complete"] = bool(turns) and all(t["trace_complete"] for t in turns)
    record["latency"]["agent_s"] = sum(t["latency"]["agent_s"] or 0 for t in turns)
    record["latency"]["scope"] = "sum of current case's Runner.run calls; excludes user wait/restart and prior cases"
    refresh_evidence(record)

async def history_turn(parent, agent, prompt, session, name):
    turn = begin_run(parent["case_id"], prompt, agent.instructions,
        max_turns=parent["configuration"]["max_turns"],
        source_recursion_limit=parent["configuration"]["source_recursion_limit"], timeout_s=120)
    turn.update({"run_id": parent["run_id"], "turn_id": uuid4().hex, "turn_name": name,
                 "session_id": session.session_id})
    await run_once(turn, agent, session=session)
    return turn


# 1.Create Agent / Smoke Test

ใช้ `Agent` และ `await Runner.run` ผ่าน helper สั้นใน Config เพื่อเก็บ error/trace ของทุก run คง prompt และข้อความคาดหมาย **DEEP_AGENTS_SDK_OK** ตามต้นฉบับโดยตรง แม้กำลังทดสอบ OpenAI SDK

In [ ]:
smoke_system_prompt = ("This is a connection smoke test. "
        "Do not call tools or delegate work. "
        "Reply with exactly: DEEP_AGENTS_SDK_OK")


In [ ]:
smoke_agent = Agent(name='Smoke Test', model=model,
    instructions=smoke_system_prompt, tools=[], model_settings=BASE_SETTINGS)
print("Agent ready:", smoke_agent.name)


In [ ]:
smoke_report = begin_run('smoke_001', 'Run the smoke test.', smoke_agent.instructions,
    case_version=None, max_turns=10, source_recursion_limit=10, timeout_s=None)

smoke_report_result = await run_once(smoke_report, smoke_agent)
show_run(smoke_report)


In [ ]:
score(smoke_report, {
    "exact_answer": smoke_report["raw_answer"] == "DEEP_AGENTS_SDK_OK",
    "no_tool_calls": smoke_report["tool_call_count"] == 0,
    "final_answer_complete": final_complete(smoke_report),
})
show_run(smoke_report)


### บันทึกผลการทดสอบ smoke

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
smoke_export_path = export_results('smoke', [smoke_report])
print("Saved:", smoke_export_path)


# 2.Mathematic calculation

ทดสอบโจทย์คำนวณตัวชี้วัดทางการเงินจากข้อมูลที่กำหนด พร้อมเครื่องมือ calculator สำหรับการคำนวณ และตรวจทั้งรูปแบบ JSON ค่าผลลัพธ์ และการใช้เครื่องมือให้ตรงตามข้อกำหนด

### Math01 (Basic) กรณีข้อมูลบริษัทเดียว

โจทย์แรกให้คำนวณการเติบโตของรายได้ อัตรากำไรสุทธิ และหนี้สินต่อสินทรัพย์ แล้วตรวจว่าคำตอบมี key และค่าตัวเลขครบถ้วน

In [ ]:
@function_tool(failure_error_function=None)
def calculator(
    operation: Literal["add", "subtract", "multiply", "divide"],
    a: float,
    b: float,
) -> float:
    """Calculate a + b, a - b, a * b, or a / b.

    Choose the operation and provide the two operands.
    For subtraction and division, operand order matters.
    """
    if operation == "add":
        return a + b
    if operation == "subtract":
        return a - b
    if operation == "multiply":
        return a * b
    if operation == "divide":
        if b == 0:
            raise ValueError("Cannot divide by zero.")
        return a / b

    raise ValueError(f"Unsupported operation: {operation}")


In [ ]:
MATH_CASE_ID = "math_001"

math_prompt = """
บริษัทสมมติ Alpha มีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท:

- รายได้ปี 2024: 1,000
- รายได้ปี 2025: 1,250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2,000

จงคำนวณ:
1. การเติบโตของรายได้ปี 2025 เทียบกับปี 2024 เป็นเปอร์เซ็นต์
2. อัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
3. หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025 เป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณ
- ใช้เฉพาะข้อมูลที่ให้
- ตอบเป็น JSON object เท่านั้น ไม่ใส่ Markdown code fence
- มีสาม keys ตามนี้ และค่าต้องเป็นตัวเลข:
  revenue_growth_pct
  net_margin_pct
  liabilities_to_assets_pct
- ค่าเปอร์เซ็นต์ให้ใช้รูปแบบ เช่น 15 สำหรับ 15%
- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง
"""


In [ ]:
math_system_prompt = ("You solve numerical tasks using the provided data. "
        "Choose the appropriate formulas yourself. "
        "Use calculator for arithmetic. "
        "Do not delegate work or use other tools for this task. "
        "Follow the user's output format exactly.")


In [ ]:
math_agent = Agent(name='Math', model=model,
    instructions=math_system_prompt, tools=[calculator], model_settings=BASE_SETTINGS)
print("Agent ready:", math_agent.name)


In [ ]:
math_report = begin_run('math_001', math_prompt, math_agent.instructions,
    case_version=None, max_turns=20, source_recursion_limit=40, timeout_s=120)

math_report_result = await run_once(math_report, math_agent)
show_run(math_report)


In [ ]:
expected = {
    "revenue_growth_pct": 25.0,
    "net_margin_pct": 12.0,
    "liabilities_to_assets_pct": 30.0,
}


In [ ]:
math_answer = math_report["answer"]
math_calls = calls_of(math_report)
score(math_report, {
    "format_passed": isinstance(math_answer, dict) and set(math_answer) == set(expected)
        and all(type(v) in (int, float) and math.isfinite(v) for v in math_answer.values()),
    **{k: matches_expected((math_answer or {}).get(k), v) if isinstance(math_answer, dict) else False
       for k, v in expected.items()},
    "only_calculator_used": bool(math_calls) and all(c["name"] == "calculator" for c in math_calls),
    "final_answer_complete": final_complete(math_report),
})
show_run(math_report)


### Math02 (Complex) กรณีขอบเขตและการตรวจความถูกต้อง

เพิ่มความซับซ้อนจากกรณีแรก โดยเปรียบเทียบตัวชี้วัดของหลายบริษัท จัดอันดับผลลัพธ์ และคัดเลือกบริษัทตามเงื่อนไขที่กำหนด นอกจากตรวจคำตอบแล้ว ยังเก็บรายละเอียดการเรียก calculator และบันทึกสาเหตุของความคลาดเคลื่อนที่พบในผลทดลอง

กลุ่มเซลล์นี้ประกอบด้วยโจทย์หลายรูปแบบและการตรวจผลตามค่าคาดหมาย เพื่อให้เห็นว่า agent จัดการข้อมูล ตัวหาร และรูปแบบผลลัพธ์ได้ถูกต้องเพียงใด

In [ ]:
MATH_002_CASE_ID = "math_002"

math_002_prompt = """
บริษัทสมมติสามแห่งมีข้อมูลจากงบรวมดังนี้
ตัวเลขทั้งหมดมีหน่วยเป็นล้านบาท และใช้หลักบัญชีเดียวกัน

Alpha:
- รายได้ปี 2024: 1000
- รายได้ปี 2025: 1250
- กำไรสุทธิปี 2025: 150
- หนี้สินรวม ณ สิ้นปี 2025: 600
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

Beta:
- รายได้ปี 2024: 800
- รายได้ปี 2025: 880
- กำไรสุทธิปี 2025: 176
- หนี้สินรวม ณ สิ้นปี 2025: 900
- สินทรัพย์รวม ณ สิ้นปี 2025: 1500

Gamma:
- รายได้ปี 2024: 1500
- รายได้ปี 2025: 1800
- กำไรสุทธิปี 2025: 180
- หนี้สินรวม ณ สิ้นปี 2025: 400
- สินทรัพย์รวม ณ สิ้นปี 2025: 2000

งานที่ต้องทำ:
1. คำนวณสามตัวชี้วัดของแต่ละบริษัทเป็นเปอร์เซ็นต์:
   - การเติบโตของรายได้ปี 2025 เทียบกับปี 2024
   - อัตรากำไรสุทธิปี 2025
   - หนี้สินรวมต่อสินทรัพย์รวม ณ สิ้นปี 2025
2. จัดอันดับบริษัท:
   - การเติบโตของรายได้: มากไปน้อย
   - อัตรากำไรสุทธิ: มากไปน้อย
   - หนี้สินรวมต่อสินทรัพย์รวม: น้อยไปมาก
3. คัดบริษัทที่ผ่านเงื่อนไขทั้งสองข้อ:
   - การเติบโตของรายได้ตั้งแต่ 20% ขึ้นไป
   - อัตรากำไรสุทธิตั้งแต่ 12% ขึ้นไป
   เรียงรายชื่อบริษัทที่ผ่านตามตัวอักษร

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณ
- ใช้เฉพาะข้อมูลที่ให้
- เปอร์เซ็นต์ใช้ตัวเลข เช่น 15 สำหรับ 15%
- ปัดผลลัพธ์สุดท้ายเป็นทศนิยมไม่เกินสองตำแหน่ง
- ตอบ JSON object เท่านั้น ไม่ใส่ Markdown code fence
- มี top-level keys เท่านั้นดังนี้:

  companies:
    object ที่มี keys Alpha, Beta, Gamma
    แต่ละบริษัทมีสาม numeric keys:
      revenue_growth_pct
      net_margin_pct
      liabilities_to_assets_pct

  rankings:
    object ที่มีสาม keys แต่ละค่าเป็น list ของชื่อบริษัท:
      revenue_growth_desc
      net_margin_desc
      liabilities_to_assets_asc

  eligible_companies:
    list ของชื่อบริษัทที่ผ่านเงื่อนไข
"""


In [ ]:
math_002_report = begin_run('math_002', math_002_prompt, math_agent.instructions,
    case_version=None, max_turns=40, source_recursion_limit=80, timeout_s=180)

math_002_report_result = await run_once(math_002_report, math_agent)
show_run(math_002_report)


In [ ]:
math_002_expected = {
    "companies": {
        "Alpha": {
            "revenue_growth_pct": 25.0,
            "net_margin_pct": 12.0,
            "liabilities_to_assets_pct": 30.0,
        },
        "Beta": {
            "revenue_growth_pct": 10.0,
            "net_margin_pct": 20.0,
            "liabilities_to_assets_pct": 60.0,
        },
        "Gamma": {
            "revenue_growth_pct": 20.0,
            "net_margin_pct": 10.0,
            "liabilities_to_assets_pct": 20.0,
        },
    },
    "rankings": {
        "revenue_growth_desc": ["Alpha", "Gamma", "Beta"],
        "net_margin_desc": ["Beta", "Alpha", "Gamma"],
        "liabilities_to_assets_asc": ["Gamma", "Alpha", "Beta"],
    },
    "eligible_companies": ["Alpha"],
}


In [ ]:
math_002_parsed = math_002_report["answer"]
actual_root = math_002_parsed if isinstance(math_002_parsed, dict) else {}
math_002_calls = calls_of(math_002_report)
score(math_002_report, {
    "top_level_structure": isinstance(math_002_parsed, dict) and set(actual_root) == set(math_002_expected),
    "all_company_metrics": matches_expected(actual_root.get("companies"), math_002_expected["companies"]),
    "all_rankings": matches_expected(actual_root.get("rankings"), math_002_expected["rankings"]),
    "eligible_companies": matches_expected(actual_root.get("eligible_companies"), math_002_expected["eligible_companies"]),
    "only_calculator_used": bool(math_002_calls) and all(c["name"] == "calculator" for c in math_002_calls),
    "final_answer_complete": final_complete(math_002_report),
})
# ตรวจ operand จาก trace ปัจจุบันเท่านั้น ไม่คัดลอกคำวิเคราะห์ที่ต้นฉบับใส่ไว้ตายตัว
math_002_report["operand_observations"] = [c for c in math_002_calls
    if isinstance(c["args"], dict) and c["args"].get("operation") == "divide" and c["args"].get("a") == 150]
print(json.dumps(math_002_report["tool_trace"], ensure_ascii=False, indent=2))
show_run(math_002_report)


### Math03 (Error handerling)

In [ ]:
math_003_prompt = """
บริษัทสมมติ Delta มีข้อมูลจากงบรวม:

- รายได้ปี 2024: 0 ล้านบาท
- รายได้ปี 2025: 1.5 พันล้านบาท
- กำไรสุทธิปี 2025: ไม่มีข้อมูล
- หนี้สินรวม ณ สิ้นปี 2025: 600 ล้านบาท
- สินทรัพย์รวม ณ สิ้นปี 2025: 2 พันล้านบาท

จง:
1. แปลงรายได้ปี 2025 เป็นล้านบาท
2. คำนวณการเติบโตของรายได้ปี 2025 เทียบปี 2024 เป็นเปอร์เซ็นต์
3. คำนวณอัตรากำไรสุทธิปี 2025 เป็นเปอร์เซ็นต์
4. คำนวณหนี้สินรวมต่อสินทรัพย์รวมเป็นเปอร์เซ็นต์

ข้อกำหนด:
- ใช้ calculator สำหรับการคำนวณและแปลงหน่วย
- ข้อมูลขาด ห้ามสมมติตัวเลขขึ้นมา
- ค่าที่คำนวณไม่ได้ให้เป็น null ไม่ใช้ 0 หรือ Infinity แทน
- ระบุ reason_code ของแต่ละอัตราส่วน:
  OK = คำนวณได้
  ZERO_DENOMINATOR = ตัวหารเป็นศูนย์
  MISSING_DATA = ข้อมูลที่จำเป็นขาด
- เปอร์เซ็นต์ใช้ตัวเลข เช่น 15 สำหรับ 15%
- ตอบ JSON เท่านั้น ไม่ใส่ Markdown code fence
- ใช้โครงสร้างดังนี้ โดยแทนที่ข้อความตัวอย่างด้วยค่าจริง:

{
  "revenue_2025_million": "ตัวเลข",
  "revenue_growth": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  },
  "net_margin": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  },
  "liabilities_to_assets": {
    "value_pct": "ตัวเลขหรือ null",
    "reason_code": "รหัส"
  }
}
"""


In [ ]:
math_003_report = begin_run('math_003', math_003_prompt, math_agent.instructions,
    case_version=None, max_turns=20, source_recursion_limit=40, timeout_s=120)

math_003_report_result = await run_once(math_003_report, math_agent)
show_run(math_003_report)


In [ ]:
math_003_expected = {
    "revenue_2025_million": 1500.0,
    "revenue_growth": {
        "value_pct": None,
        "reason_code": "ZERO_DENOMINATOR",
    },
    "net_margin": {
        "value_pct": None,
        "reason_code": "MISSING_DATA",
    },
    "liabilities_to_assets": {
        "value_pct": 30.0,
        "reason_code": "OK",
    },
}


In [ ]:
math_003_calls = calls_of(math_003_report)
score(math_003_report, {
    **{k: matches_expected(math_003_report["answer"].get(k), v)
       if isinstance(math_003_report["answer"], dict) else False for k, v in math_003_expected.items()},
    "exact_structure_and_values": matches_expected(math_003_report["answer"], math_003_expected),
    "only_calculator_used": bool(math_003_calls) and all(c["name"] == "calculator" for c in math_003_calls),
    "final_answer_complete": final_complete(math_003_report),
})
show_run(math_003_report)


### บันทึกผลการทดสอบ math

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
math_export_path = export_results('math', [math_report, math_002_report, math_003_report])
print("Saved:", math_export_path)


# 3.MCP

คง FastMCP fixtures, tool names, arguments และ error scenarios เดิม ใช้ `Agent(mcp_servers=[...])` กับ `MCPServerStreamableHttp` ของ OpenAI SDK โดยตรง

ต้นฉบับใช้ in-memory transport ผ่าน MCPAdapter; SDK รุ่นที่ตรวจไม่มี concrete in-memory server adapter จึงเปิด FastMCP บน localhost ด้วย Streamable HTTP และพอร์ตว่างอัตโนมัติ นี่เป็น MCP protocol จริง ไม่ใช่ function calls ตรงเข้า fixture แยก setup/agent/cleanup latency; ไม่เปิด API retry หรือ MCP transport retry เพื่อช่วยเคส retry

In [ ]:
import socket
import uvicorn
from fastmcp import FastMCP
from agents.mcp import MCPServerStreamableHttp

@asynccontextmanager
async def local_mcp(server):
    # bind port 0 ล่วงหน้า ป้องกัน race ของการหา free port
    sock = socket.socket()
    sock.bind(("127.0.0.1", 0))
    sock.listen(128)
    port = sock.getsockname()[1]
    app = server.http_app(path="/mcp", transport="streamable-http", stateless_http=True)
    http_server = uvicorn.Server(uvicorn.Config(app, log_level="error", access_log=False))
    task = asyncio.create_task(http_server.serve(sockets=[sock]))
    try:
        async with asyncio.timeout(15):
            while not http_server.started:
                if task.done():
                    await task
                    raise RuntimeError("MCP HTTP server stopped before startup")
                await asyncio.sleep(0.01)
        # connect และ cleanup ของ MCP client อยู่ใน asyncio task เดียวกัน
        async with MCPServerStreamableHttp(
            name="Finance Benchmark", params={"url": f"http://127.0.0.1:{port}/mcp"},
            cache_tools_list=True, client_session_timeout_seconds=30,
            max_retry_attempts=0, failure_error_function=None,
        ) as connection:
            yield connection
    finally:
        http_server.should_exit = True
        try:
            await asyncio.wait_for(task, timeout=10)
        finally:
            if not task.done():
                task.cancel()
            with suppress(asyncio.CancelledError):
                await task
            sock.close()

async def run_mcp_once(record, server, server_log):
    started = time.perf_counter()
    agent_finished = None
    record["transport"] = "streamable_http_localhost"
    record["source_transport"] = "in_memory"
    record["discovered_tools"] = []
    record["configuration"]["mcp_max_retry_attempts"] = 0
    try:
        async with asyncio.timeout(120):
            async with local_mcp(server) as connection:
                discovered = await connection.list_tools()
                record["discovered_tools"] = [t.name for t in discovered]
                record["discovered_tool_schemas"] = [jsonable(t) for t in discovered]
                mcp_agent = Agent(name="Finance MCP", model=model,
                    instructions=record["system_prompt"], mcp_servers=[connection],
                    model_settings=BASE_SETTINGS)
                record["latency"]["setup_s"] = time.perf_counter() - started
                await run_once(record, mcp_agent)
                agent_finished = time.perf_counter()
    except Exception as exc:
        record["runtime_status"] = "failed"
        record["trace_complete"] = False
        record["error"] = {"type": type(exc).__name__, "message": str(exc)}
        if record["latency"]["agent_s"] is None:
            record["latency"]["setup_s"] = time.perf_counter() - started
    finally:
        record["server_trace"] = deepcopy(server_log)
        record["server_tool_calls_observed"] = len(server_log)
        record["latency"]["cleanup_s"] = time.perf_counter() - agent_finished if agent_finished else None
        record["latency"]["total_s"] = time.perf_counter() - started
        refresh_evidence(record)
        record["latency_s"] = record["latency"]["total_s"]


In [ ]:
finance_mcp = FastMCP("Finance Benchmark")

mcp_server_log = []

COMPANY = {
    "company_id": "cmp_a17",
    "company_name": "Alpha",
}

STATEMENT = {
    "company_id": "cmp_a17",
    "year": 2025,
    "revenue_million": 1375,
    "net_profit_million": 165,
    "source_id": "fixture_alpha_2025_v1",
}


@finance_mcp.tool()
def search_company(query: str) -> dict:
    """Search company names and return matching company IDs and names."""
    normalized = query.strip().casefold()

    matches = (
        [COMPANY.copy()]
        if normalized and normalized in COMPANY["company_name"].casefold()
        else []
    )

    output = {"companies": matches}

    mcp_server_log.append({
        "tool": "search_company",
        "arguments": {"query": query},
        "output": output,
    })

    return output


@finance_mcp.tool()
def get_financial_statement(company_id: str, year: int) -> dict:
    """Retrieve a financial statement using a company ID and year."""
    found = (
        company_id == STATEMENT["company_id"]
        and year == STATEMENT["year"]
    )

    output = (
        STATEMENT.copy()
        if found
        else {"error": "STATEMENT_NOT_FOUND"}
    )

    mcp_server_log.append({
        "tool": "get_financial_statement",
        "arguments": {"company_id": company_id, "year": year},
        "output": output,
    })

    return output


print("MCP server ready.")


In [ ]:
mcp_system_prompt = ("Answer using the provided financial data tools. "
                "Resolve the company before retrieving its statement. "
                "Do not guess identifiers or financial figures. "
                "Do not delegate or use other tools. "
                "Follow the requested JSON format.")


### MCP01 (ดูงบบริษัท)

กรณีพื้นฐานสำหรับเรียกเครื่องมือ MCP เพื่อขอข้อมูลงบ ตรวจรูปแบบคำตอบและ trace ที่เซิร์ฟเวอร์ได้รับ

In [ ]:
mcp_prompt = """
หาข้อมูลงบปี 2025 ของบริษัท Alpha จากเครื่องมือที่ให้

ตอบ JSON object เท่านั้น มี keys:
- company_id
- year
- revenue_million
- net_profit_million
- source_id

ใช้ข้อมูลจากผลเครื่องมือจริง ห้ามเดารหัสบริษัทหรือตัวเลข
ไม่ใส่ Markdown code fence
"""


In [ ]:
mcp_report = begin_run("mcp_001", mcp_prompt, mcp_system_prompt,
    max_turns=20, source_recursion_limit=40, timeout_s=120)
mcp_server_log.clear()
await run_mcp_once(mcp_report, finance_mcp, mcp_server_log)
show_run(mcp_report)


In [ ]:
# Dependency ตรวจจาก server log ตามต้นฉบับ
chain_passed = False
for index, event in enumerate(mcp_report["server_trace"]):
    if event["tool"] == "search_company":
        found_ids = {row["company_id"] for row in event["output"]["companies"]}
        for later in mcp_report["server_trace"][index + 1:]:
            if (later["tool"] == "get_financial_statement"
                and later["arguments"]["company_id"] in found_ids
                and later["arguments"]["year"] == 2025
                and "error" not in later["output"]):
                chain_passed = True
expected_tools = {"search_company", "get_financial_statement"}
mcp_calls = calls_of(mcp_report)
score(mcp_report, {
    "tools_discovered": set(mcp_report["discovered_tools"]) == expected_tools,
    "company_id_forwarded": chain_passed,
    "answer_correct": isinstance(mcp_report["answer"], dict) and mcp_report["answer"] == STATEMENT,
    "only_expected_tools": bool(mcp_calls) and all(c["name"] in expected_tools for c in mcp_calls),
    "final_answer_complete": final_complete(mcp_report),
})
print(json.dumps(mcp_report["server_trace"], ensure_ascii=False, indent=2))
show_run(mcp_report)


### MCP02 (หาบริษัทที่หา ไม่ได้มีตัวตน)

กรณีทดสอบถัดไปตรวจการทำงานกับคำขออีกแบบ โดยเก็บ trace ของกรณีก่อนหน้าไว้ก่อนเริ่มการทดลองใหม่ เพื่อแยกผลของแต่ละเคสได้ชัดเจน

In [ ]:
mcp_002_prompt = """
ค้นหาบริษัท Omega จากเครื่องมือที่ให้ และดึงงบปี 2025 หากพบบริษัท

ข้อกำหนด:
- ห้ามเดา company ID หรือใช้ข้อมูลของบริษัทอื่นแทน
- หากไม่พบบริษัท ให้รายงานว่าไม่พบและไม่เรียกดึงงบ
- ตอบ JSON object เท่านั้น ไม่ใส่ Markdown code fence

โครงสร้างคำตอบ:
{
  "status": "found หรือ not_found",
  "company_name": "Omega",
  "statement": "ข้อมูลงบจากเครื่องมือ หรือ null หากไม่พบ"
}
"""


In [ ]:
mcp_002_report = begin_run("mcp_002", mcp_002_prompt, mcp_system_prompt,
    max_turns=20, source_recursion_limit=40, timeout_s=120)
mcp_server_log.clear()
await run_mcp_once(mcp_002_report, finance_mcp, mcp_server_log)
show_run(mcp_002_report)


In [ ]:
expected_002 = {
    "status": "not_found",
    "company_name": "Omega",
    "statement": None,
}


In [ ]:
trace_002 = mcp_002_report["server_trace"]
calls_002 = calls_of(mcp_002_report)
score(mcp_002_report, {
    "tools_discovered": set(mcp_002_report["discovered_tools"]) == expected_tools,
    "received_empty_search_result": any(e["tool"] == "search_company" and e["output"]["companies"] == [] for e in trace_002),
    "did_not_fetch_statement": not any(e["tool"] == "get_financial_statement" for e in trace_002),
    "only_search_tool_used": bool(calls_002) and all(c["name"] == "search_company" for c in calls_002),
    "answer_correct": mcp_002_report["answer"] == expected_002,
    "final_answer_complete": final_complete(mcp_002_report),
})
show_run(mcp_002_report)


### MCP03 (ถ้าหากว่ามี Error ให้ทำการ Retry)

โค้ดชุดนี้ทดสอบว่า agent จะ ลองเรียกเครื่องมือซ้ำหนึ่งครั้งเมื่อได้รับ error ที่ระบุว่า retry ได้ แล้วนำข้อมูลที่ได้สำเร็จมาตอบหรือไม่

In [ ]:
def build_retry_mcp():
    server = FastMCP("Finance Retry Benchmark")
    trace = []
    state = {"statement_attempts": 0}

    @server.tool()
    def search_company(query: str) -> dict:
        """Search company names and return matching company IDs and names."""
        normalized = query.strip().casefold()

        output = {
            "companies": (
                [COMPANY.copy()]
                if normalized
                and normalized in COMPANY["company_name"].casefold()
                else []
            )
        }

        trace.append({
            "tool": "search_company",
            "arguments": {"query": query},
            "output": output,
        })

        return output

    @server.tool()
    def get_financial_statement(company_id: str, year: int) -> dict:
        """Retrieve a financial statement by company ID and year.

        On failure, the response includes an error code and retryable flag.
        """
        valid_request = (
            company_id == STATEMENT["company_id"]
            and year == STATEMENT["year"]
        )

        if not valid_request:
            output = {
                "error": {
                    "code": "STATEMENT_NOT_FOUND",
                    "retryable": False,
                }
            }
        else:
            state["statement_attempts"] += 1

            if state["statement_attempts"] == 1:
                output = {
                    "error": {
                        "code": "TEMPORARY_UNAVAILABLE",
                        "retryable": True,
                    }
                }
            else:
                output = STATEMENT.copy()

        trace.append({
            "tool": "get_financial_statement",
            "arguments": {"company_id": company_id, "year": year},
            "output": output,
        })

        return output

    return server, trace


In [ ]:
mcp_003_prompt = """
หาข้อมูลงบปี 2025 ของบริษัท Alpha จากเครื่องมือที่ให้

นโยบายจัดการข้อผิดพลาด:
- หาก tool คืน error ที่ retryable เป็น true ให้ลองคำขอนั้นใหม่
  ได้ไม่เกินหนึ่งครั้ง
- หากยังไม่สำเร็จ ให้รายงานข้อผิดพลาดตามจริง
- ห้ามเดารหัสบริษัทหรือตัวเลข

เมื่อสำเร็จ ตอบ JSON object เท่านั้น มี keys:
- company_id
- year
- revenue_million
- net_profit_million
- source_id

ไม่ใส่ Markdown code fence
"""


In [ ]:
mcp_003_report = begin_run("mcp_003", mcp_003_prompt, mcp_system_prompt,
    max_turns=20, source_recursion_limit=40, timeout_s=120)
retry_server, mcp_003_trace = build_retry_mcp()  # reset fixture ทุก run
await run_mcp_once(mcp_003_report, retry_server, mcp_003_trace)
show_run(mcp_003_report)


In [ ]:
expected_arguments = {
    "company_id": COMPANY["company_id"],
    "year": 2025,
}


In [ ]:
statement_events = [e for e in mcp_003_report["server_trace"] if e["tool"] == "get_financial_statement"]
resolved_ids = set()
company_id_forwarded = True
for event in mcp_003_report["server_trace"]:
    if event["tool"] == "search_company":
        resolved_ids.update(c["company_id"] for c in event["output"]["companies"])
    elif event["tool"] == "get_financial_statement" and event["arguments"]["company_id"] not in resolved_ids:
        company_id_forwarded = False
calls_003 = calls_of(mcp_003_report)
score(mcp_003_report, {
    "tools_discovered": set(mcp_003_report["discovered_tools"]) == expected_tools,
    "company_id_forwarded": bool(statement_events) and company_id_forwarded,
    "retried_once_and_recovered": len(statement_events) == 2
        and all(e["arguments"] == expected_arguments for e in statement_events)
        and statement_events[0]["output"].get("error") == {"code": "TEMPORARY_UNAVAILABLE", "retryable": True}
        and statement_events[1]["output"] == STATEMENT,
    "answer_correct": mcp_003_report["answer"] == STATEMENT,
    "only_expected_tools": bool(calls_003) and all(c["name"] in expected_tools for c in calls_003),
    "final_answer_complete": final_complete(mcp_003_report),
})
mcp_003_report["failure_mode"] = "retryable_error_in_tool_result"
show_run(mcp_003_report)


### บันทึกผลการทดสอบ mcp

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
mcp_export_path = export_results('mcp', [mcp_report, mcp_002_report, mcp_003_report])
print("Saved:", mcp_export_path)


# 4.Web search

ทดสอบความสามารถในการค้นคว้าข้อมูลบนเว็บ โดยตรวจทั้งการใช้เครื่องมือค้นหาและการเปิดอ่านแหล่งข้อมูล รวมถึงความครบถ้วนของคำตอบและแหล่งอ้างอิง

ใช้ `WebSearchTool()` ผ่าน OpenAI hosted web เท่านั้น คง `include=["web_search_call.action.sources"]` จากต้นฉบับ

นับ search/open-page จาก `raw_responses[].output[].type == "web_search_call"` และ `action.type` โดยตรง Citation และ search sources ใช้เป็นข้อมูลสำหรับ review เท่านั้น ไม่ใช้เป็นหลักฐานว่าเปิดหน้าแล้ว

`status=completed` ของ tool action ไม่ยืนยันว่า HTTP fetch สำเร็จหรือเนื้อหารองรับคำตอบเสมอไป จึงเก็บสองเรื่องแยกกัน และให้ review หลักฐานเนื้อหาเป็น `null` จนกว่าจะตรวจได้จริง ไม่วนเรียกโมเดลใหม่เพื่อให้ผ่าน

In [ ]:
native_web_system_prompt = """
You are a financial research assistant.

Search the web for official sources, then open relevant URLs
found during search using the web tool's page-opening capability.
Read the opened sources before analyzing the figures.
Do not rely only on search snippets.

For comparisons between companies, open supporting sources
for each company.

Prefer official company publications or filings submitted to the SEC.
Distinguish fiscal years from calendar years and quarters.
Cite sources supporting your claims.
If you cannot open a source or evidence is insufficient,
state the limitation. Do not invent data or claim a page was read.

Do not delegate or use other tools for this task.
Answer in Thai.
""".strip()


In [ ]:
web_client = AsyncOpenAI(max_retries=0, timeout=90)
web_model = OpenAIResponsesModel(model=MODEL, openai_client=web_client)
WEB_SETTINGS = ModelSettings(response_include=["web_search_call.action.sources"],
    preserve_raw_usage=True, retry=ModelRetrySettings(max_retries=0))
native_web_agent = Agent(name="Hosted Web Research", model=web_model,
    instructions=native_web_system_prompt, tools=[WebSearchTool()], model_settings=WEB_SETTINGS)


In [ ]:
def source_owner(url):
    # Hostname ตรงจริงเท่านั้น; CDN/SEC URL ที่ไม่ระบุตัวเจ้าของชัดเจนต้องรอตรวจ
    try:
        parsed = urlsplit(url)
        host = (parsed.hostname or "").lower()
        if parsed.scheme not in {"http", "https"}:
            return None
    except (ValueError, TypeError):
        return None
    if host == "microsoft.com" or host.endswith(".microsoft.com") or host == "microsoft.gcs-web.com":
        return "Microsoft"
    if host == "abc.xyz" or host.endswith(".abc.xyz"):
        return "Alphabet"
    # SEC CIK เป็น company identifier ใน URL (ไม่ใช้ชื่อที่เดาจาก citation title)
    if host in {"sec.gov", "www.sec.gov"}:
        match = re.match(r"/Archives/edgar/data/(\d+)/", parsed.path, re.IGNORECASE)
        if match:
            return {789019: "Microsoft", 1652044: "Alphabet"}.get(int(match.group(1)))
    return None

def page_identity(url):
    try:
        p = urlsplit(url)
        return (p.hostname.lower(), p.path.rstrip("/") or "/") if p.hostname and p.scheme in {"http", "https"} else None
    except (ValueError, TypeError):
        return None

def inspect_web(record, companies):
    raw_items = [item for r in record["raw_responses"] for item in r.get("output", [])]
    web_calls = [item for item in raw_items if item.get("type") == "web_search_call"]
    actions = Counter(item.get("action", {}).get("type", "unknown") for item in web_calls)
    opened = [item for item in web_calls if item.get("action", {}).get("type") == "open_page"]
    urls = [item.get("action", {}).get("url") for item in opened]
    ownership = [{"url": u, "owner": source_owner(u), "action_status": item.get("status"),
                  "action_id": item.get("id"), "content_support_verified": None}
                 for u, item in zip(urls, opened)]
    unknown_owner = any(row["owner"] is None for row in ownership)
    owner_checks = {f"opened_source_{company}": True if any(row["owner"] == company for row in ownership)
                    else None if unknown_owner else False for company in companies}
    unique_pages = {p for u in urls if (p := page_identity(u)) is not None}
    record.update({"action_counts": dict(actions), "opened_urls": urls,
                   "source_ownership": ownership, "web_actions": deepcopy(web_calls),
                   "search_provider": "openai", "search_mode": "provider_native",
                   "fetch_mode": "provider_native_open_page"})
    checks = {
        "web_search_observed": actions["search"] > 0,
        "web_fetch_observed": actions["open_page"] > 0,
        "citations_observed": bool(record["citations"]),
        "no_other_tool_calls": not calls_of(record) and not any(
            item.get("type", "").endswith("_call") and item.get("type") != "web_search_call" for item in raw_items),
        "no_invalid_tool_calls": not any(parse_json(c.get("arguments")) is None for c in raw_items if c.get("type") == "function_call"),
        "final_answer_complete": final_complete(record), **owner_checks,
    }
    if len(companies) == 2:
        checks["at_least_two_distinct_pages_opened"] = len(unique_pages) >= 2
    score(record, checks)
    # Unknown owner เป็น pending และให้ผู้ตรวจยืนยันจาก URL/เอกสารจริง; สอง Microsoft URLs จะเป็น false สำหรับ Alphabet
    rubric = {key: "ยืนยันบริษัทเจ้าของเอกสารจาก open_page URL/หลักฐานจริง; ห้ามใช้ citation แทน open_page"
              for key, value in owner_checks.items() if value is None}
    rubric.update({
        "opened_successfully_and_content_supports_answer": "ตรวจแต่ละบริษัทว่าเปิดสำเร็จและเนื้อหาที่เปิดรองรับตัวเลข; completed/citation อย่างเดียวไม่เพียงพอ",
        "correct_annual_financial_figures_units_periods": "งบรวมทั้งปี หน่วยล้าน USD วันเริ่ม/สิ้นสุดปีถูกต้องตามโจทย์ (Web01 ต้องระบุวันสิ้นสุด)",
        "correct_formulas_substitution_and_rounding": "สูตร การแทนค่า และผลเปอร์เซ็นต์สองตำแหน่งถูกต้อง",
        "citations_support_claims": "แหล่งอ้างอิงรองรับแต่ละ claim/บริษัท ไม่มีการเดาหรือปนไตรมาส/segment",
        "thai_table_conclusion_and_limitations": "ครบตาราง สูตร ข้อสรุปไทย; Web02 อธิบายช่วงปีต่างกันและไม่สรุปว่าดีกว่าทุกด้าน",
    })
    add_review(record, rubric)
    return record


### Websearch01 (ค้นหาและเปรียบเทียบ)

เปรียบเทียบรายได้ Microsoft FY2024 กับ FY2023 โดยค้นเว็บและเปิดอ่านแหล่งข้อมูลทางการก่อนตอบ

In [ ]:
native_web_prompt = """
ค้นเว็บเพื่อเปรียบเทียบรายได้รวมของ Microsoft
สำหรับปีงบประมาณ 2024 และ 2023

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของ Microsoft
- เปิดอ่าน URL ที่พบจากการค้นหาอย่างน้อยหนึ่งหน้า
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

ข้อกำหนด:
1. ใช้รายได้รวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
2. แสดงตัวเลขเป็นล้านดอลลาร์สหรัฐ
3. ระบุวันสิ้นสุดปีงบประมาณของแต่ละปี
4. คำนวณอัตราการเติบโต พร้อมแสดงสูตรและการแทนค่า
5. แสดงอัตราการเติบโตทศนิยมสองตำแหน่ง
6. สรุปภาษาไทยสั้น ๆ ว่ารายได้เปลี่ยนแปลงอย่างไร
7. อ้างอิงแหล่งข้อมูลที่รองรับตัวเลข
8. หากเปิดแหล่งข้อมูลไม่ได้หรือข้อมูลไม่พอ ให้ระบุข้อจำกัด
9. ห้ามเดาตัวเลข

ตอบเป็นตาราง ตามด้วยสูตร การคำนวณ และข้อสรุป
ต้องค้นเว็บและเปิดอ่านแหล่งข้อมูลจริงก่อนตอบ
""".strip()


In [ ]:
native_web_report = begin_run('web_native_001', native_web_prompt, native_web_agent.instructions,
    case_version='2.0', max_turns=10, source_recursion_limit=30, timeout_s=180)
native_web_report["configuration"]["api_timeout_s"] = 90
native_web_report_result = await run_once(native_web_report, native_web_agent)
show_run(native_web_report)


In [ ]:
inspect_web(native_web_report, ["Microsoft"])
display(Markdown(native_web_report["raw_answer"] or "*ไม่มีคำตอบสุดท้าย*"))
print(json.dumps(native_web_report["source_ownership"], ensure_ascii=False, indent=2))
show_run(native_web_report)
web_001_review_path = save_review_template(native_web_report)
print("Review template:", web_001_review_path)
# หลังตรวจและแก้ไฟล์ review แล้ว: apply_review(native_web_report, web_001_review_path)


### Websearch02 (เปรียบเทียบอ้างอิง)

เปรียบเทียบ Microsoft กับ Alphabet FY2024
โดยค้นเว็บและเปิดอ่านแหล่งข้อมูลของทั้งสองบริษัทก่อนตอบ

In [ ]:
web_002_prompt = """
ค้นเว็บเพื่อเปรียบเทียบ Microsoft กับ Alphabet
โดยใช้ผลประกอบการรวมทั้งปีงบประมาณ 2024 ของแต่ละบริษัท

ขั้นตอนที่ต้องทำ:
- ค้นหาแหล่งข้อมูลทางการของบริษัท หรือเอกสารที่ยื่นต่อ SEC
- เปิดอ่านแหล่งข้อมูลของแต่ละบริษัท อย่างน้อยบริษัทละหนึ่ง URL
  รวมอย่างน้อยสอง URL ที่แตกต่างกัน
- ตรวจตัวเลขจากเนื้อหาที่เปิดอ่าน ไม่ใช้เพียงข้อความย่อผลค้นหา

สิ่งที่ต้องหา:
1. วันเริ่มต้นและวันสิ้นสุดปีงบประมาณ
2. รายได้รวม
3. กำไรจากการดำเนินงาน (Operating income)
4. Operating margin ที่คำนวณจากตัวเลขข้างต้น

ข้อกำหนด:
- ใช้ข้อมูล GAAP จากงบรวมทั้งปี ไม่ใช้รายไตรมาสหรือราย segment
- ใช้แหล่งข้อมูลทางการของบริษัท หรือเอกสารที่บริษัทยื่นต่อ SEC
- แสดงจำนวนเงินเป็นล้านดอลลาร์สหรัฐ
- คำนวณ Operating margin เป็นเปอร์เซ็นต์ ทศนิยมสองตำแหน่ง
- แสดงสูตรและการแทนค่าของแต่ละบริษัท
- อ้างอิงให้ชัดว่าแต่ละแหล่งรองรับบริษัทใด
- ตรวจว่าช่วงปีงบประมาณตรงกันหรือไม่ และอธิบายผลต่อการเปรียบเทียบ
- สรุปว่าบริษัทใดมี Operating margin สูงกว่าในงบที่นำมาเทียบ
- ไม่สรุปว่าบริษัทนั้นดีกว่าทุกด้านจากตัวชี้วัดนี้เพียงตัวเดียว
- หากเปิดแหล่งข้อมูลไม่ได้หรือข้อมูลไม่พอ ให้ระบุข้อจำกัด
- ห้ามเดาตัวเลข

ตอบภาษาไทย ประกอบด้วย:
1. ตารางเปรียบเทียบ
2. สูตรและการคำนวณ
3. ข้อสรุปและข้อจำกัดสั้น ๆ

ต้องค้นเว็บและเปิดอ่านแหล่งข้อมูลจริงก่อนตอบ
""".strip()


In [ ]:
web_002_report = begin_run('web_native_002', web_002_prompt, native_web_agent.instructions,
    case_version='2.0', max_turns=10, source_recursion_limit=30, timeout_s=180)
web_002_report["configuration"]["api_timeout_s"] = 90
web_002_report_result = await run_once(web_002_report, native_web_agent)
show_run(web_002_report)


In [ ]:
inspect_web(web_002_report, ["Microsoft", "Alphabet"])
display(Markdown(web_002_report["raw_answer"] or "*ไม่มีคำตอบสุดท้าย*"))
print(json.dumps(web_002_report["source_ownership"], ensure_ascii=False, indent=2))
show_run(web_002_report)
web_002_review_path = save_review_template(web_002_report)
print("Review template:", web_002_review_path)
# หลังตรวจและแก้ไฟล์ review แล้ว: apply_review(web_002_report, web_002_review_path)
# ไม่มีการนำข้อสรุป "ไม่พบ Alphabet" ของรอบเก่ามาใช้ และไม่มี rerun-until-pass


### บันทึกผลการทดสอบ web_native

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
web_native_export_path = export_results('web_native', [native_web_report, web_002_report])
print("Saved:", web_native_export_path)


# 5 Planning

ทดสอบงานที่ต้องวางแผนและทำหลายขั้นตอน โดยกำหนดเครื่องมือและข้อกำหนดใน prompt จากนั้นบันทึกผลการทำงาน ตรวจลำดับการเรียกเครื่องมือ และประเมินว่าทำงานครบตามเป้าหมายหรือไม่

### Config: Custom `write_todos`

OpenAI Agents SDK ไม่มี TodoListMiddleware แบบ Deep Agents จึงเพิ่ม **custom function tool** ชื่อ `write_todos` รับ `todos: [{content, status}]`, status = `pending | in_progress | completed` แทนรายการทั้งชุดและคืนข้อความรูปแบบเดิม เก็บ state ใน run context และ log ทุก update

นำ description และ system prompt ของ TodoListMiddleware รุ่น `langchain==1.4.3` มาใช้เป็นข้อมูลข้อความเท่านั้น ไม่มีการ import/รัน LangChain หรือ LangGraph และบันทึก authored system instruction เดิม เทียบข้อความใหม่ พร้อมเหตุผลใน run metadata

In [ ]:
PLANNING_DATA = {
    "co-d81": {
        "company_id": "co-d81",
        "company_name": "Delta",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 1000,
        "revenue_current": 1300,
        "net_profit_current": 156,
        "liabilities": 540,
        "assets": 1800,
        "source_id": "fixture_delta_v1",
    },
    "co-e42": {
        "company_id": "co-e42",
        "company_name": "Echo",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 800,
        "revenue_current": 960,
        "net_profit_current": 192,
        "liabilities": 900,
        "assets": 1500,
        "source_id": "fixture_echo_v1",
    },
    "co-f63": {
        "company_id": "co-f63",
        "company_name": "Foxtrot",
        "year": 2025,
        "previous_year": 2024,
        "unit": "million_thb",
        "revenue_previous": 1500,
        "revenue_current": 1800,
        "net_profit_current": None,
        "liabilities": 400,
        "assets": 2000,
        "source_id": "fixture_foxtrot_v1",
    },
}


In [ ]:
WRITE_TODOS_TOOL_DESCRIPTION = """Use this tool to create and manage a structured task list for your current work session. This helps you track progress and organize complex tasks.

Only use this tool if you think it will be helpful in staying organized. If the user's request is trivial and takes less than 3 steps, it is better to NOT use this tool and just do the task directly.

## When to Use This Tool

Use this tool in these scenarios:

1. Complex multi-step tasks - When a task requires 3 or more distinct steps or actions
2. Non-trivial and complex tasks - Tasks that require careful planning or multiple operations
3. User explicitly requests todo list - When the user directly asks you to use the todo list
4. User provides multiple tasks - When users provide a list of things to be done (numbered or comma-separated)
5. The plan may need future revisions or updates based on results from the first few steps

## How to Use This Tool

1. When you start working on a task - Mark it as in_progress BEFORE beginning work.
2. After completing a task - Mark it as completed and add any new follow-up tasks discovered during implementation.
3. You can also update future tasks, such as deleting them if they are no longer necessary, or adding new tasks that are necessary. Don't change previously completed tasks.
4. You can make several updates to the todo list at once. For example, when you complete a task, you can mark the next task you need to start as in_progress.

## When NOT to Use This Tool

It is important to skip using this tool when:
1. There is only a single, straightforward task
2. The task is trivial and tracking it provides no benefit
3. The task can be completed in less than 3 trivial steps
4. The task is purely conversational or informational

## Task States and Management

1. **Task States**: Use these states to track progress:
    - pending: Task not yet started
    - in_progress: Currently working on (you can have multiple tasks in_progress at a time if they are not related to each other and can be run in parallel)
    - completed: Task finished successfully

2. **Task Management**:
    - Update task status in real-time as you work
    - Mark tasks complete IMMEDIATELY after finishing (don't batch completions)
    - Complete current tasks before starting new ones
    - Remove tasks that are no longer relevant from the list entirely
    - IMPORTANT: When you write this todo list, you should mark your first task (or tasks) as in_progress immediately!.
    - IMPORTANT: Unless all tasks are completed, you should always have at least one task in_progress.

3. **Task Completion Requirements**:
    - ONLY mark a task as completed when you have FULLY accomplished it
    - If you encounter errors, blockers, or cannot finish, keep the task as in_progress
    - When blocked, create a new task describing what needs to be resolved
    - Never mark a task as completed if:
        - There are unresolved issues or errors
        - Work is partial or incomplete
        - You encountered blockers that prevent completion
        - You couldn't find necessary resources or dependencies
        - Quality standards haven't been met

4. **Task Breakdown**:
    - Create specific, actionable items
    - Break complex tasks into smaller, manageable steps
    - Use clear, descriptive task names

Being proactive with task management ensures you complete all requirements successfully
Remember: If you only need to make a few tool calls to complete a task, and it is clear what you need to do, it is better to just do the task directly and NOT call this tool at all.

## When You Finish

`write_todos` tracks your work; it does not deliver the answer. Whatever the user asked for — computations, summaries, comparisons, data — must appear as text content in a message after your final `write_todos` call. Marking the last todo complete is not itself an answer to the user."""

WRITE_TODOS_SYSTEM_PROMPT = """## `write_todos`

You have access to the `write_todos` tool to help you manage and plan complex objectives.
Use this tool for complex objectives to ensure that you are tracking each necessary step.
This tool is very helpful for planning complex objectives, and for breaking down these larger complex objectives into smaller steps.

It is critical that you mark todos as completed as soon as you are done with a step. Do not batch up multiple steps before marking them as completed.
For simple objectives that only require a few steps, it is better to just complete the objective directly and NOT use this tool.
Writing todos takes time and tokens, use it when it is helpful for managing complex many-step problems! But not for simple few-step requests.

## Important To-Do List Usage Notes to Remember

- The `write_todos` tool should never be called multiple times in parallel.
- Don't be afraid to revise the To-Do list as you go. New information may reveal new tasks that need to be done, or old tasks that are irrelevant.

## Finishing a task

When you finish all work, write your final answer in the message AFTER your last `write_todos` call — not in the same turn as that call. Start the final message with the substantive content the user asked for — the data, computation, summary, or analysis. The user wants the result, not confirmation that the work is done."""


In [ ]:
from typing_extensions import TypedDict

class Todo(TypedDict):
    content: str
    status: Literal["pending", "in_progress", "completed"]

@function_tool(description_override=WRITE_TODOS_TOOL_DESCRIPTION, failure_error_function=None)
def write_todos(ctx: RunContextWrapper[dict], todos: list[Todo]) -> str:
    """Create and manage a structured task list for your current work session."""
    # เหมือน middleware: หลาย write_todos ใน model response เดียวถูกปฏิเสธทั้งหมด
    if ctx.context.get("write_todos_calls_in_response", 0) > 1:
        return ("Error: The `write_todos` tool should never be called multiple times "
                "in parallel. Please call it only once per model invocation to update "
                "the todo list.")
    ctx.context["todos"] = deepcopy(todos)
    return f"Updated todo list to {todos}"


In [ ]:
@function_tool(failure_error_function=None)
def list_companies() -> list[dict]:
    """List available companies with their IDs and names."""
    return [
        {
            "company_id": row["company_id"],
            "company_name": row["company_name"],
        }
        for row in PLANNING_DATA.values()
    ]

@function_tool(failure_error_function=None)
def read_company_financials(company_id: str) -> dict:
    """Read a company's annual financial data.

    A null value means the data is unavailable.
    """
    row = PLANNING_DATA.get(company_id)

    if row is None:
        return {"error": "COMPANY_NOT_FOUND"}

    return deepcopy(row)

@function_tool(failure_error_function=None)
def calculate_percentage(
    numerator: float,
    denominator: float,
    mode: Literal["ratio", "growth"],
) -> dict:
    """Calculate a percentage.

    ratio: numerator / denominator * 100.
    growth: (numerator - denominator) / denominator * 100;
            numerator is the current value, denominator the previous value.
    """
    if denominator == 0:
        return {"value_pct": None, "error": "ZERO_DENOMINATOR"}

    if mode == "ratio":
        value = numerator / denominator * 100
    elif mode == "growth":
        value = (numerator - denominator) / denominator * 100
    else:
        raise ValueError("Unsupported mode")

    return {"value_pct": round(value, 2)}


### Planning01

In [ ]:
planning_prompt = """
เปรียบเทียบบริษัททั้งหมดที่มีในระบบ โดยใช้ข้อมูลปี 2025
ในสามด้าน:
- การเติบโตของรายได้เทียบปีก่อน
- อัตรากำไรสุทธิ
- หนี้สินรวมต่อสินทรัพย์รวม

ระบุว่าบริษัทใดมีข้อมูลครบสำหรับคำนวณทั้งสามตัวชี้วัด
และบริษัทใดยังขาดข้อมูลอะไร

ข้อกำหนด:
- สร้างแผนงานของคุณผ่าน write_todos ก่อนเริ่มเรียก tools ดึงข้อมูล
- เลือกงานย่อยและลำดับการทำงานเอง
- อัปเดตสถานะ To-do ระหว่างทำงานให้ตรงกับสิ่งที่ทำจริง
- ใช้ calculate_percentage สำหรับการคำนวณ
- ข้อมูลขาดให้แสดง N/A และอธิบาย ห้ามสมมติตัวเลข
- ข้อมูลครบ หมายถึงมีข้อมูลสำหรับคำนวณทั้งสามตัวชี้วัด
- อ้างอิง source_id สำหรับข้อมูลของแต่ละบริษัท
- ตอบภาษาไทยเป็นตาราง พร้อมข้อสรุปสั้น ๆ
"""


In [ ]:
planning_system_prompt = ("Complete the user's analysis using the available tools. "
        "Maintain an actionable task list reflecting actual progress. "
        "Use only the supplied financial data. "
        "Treat missing values as unavailable, not as zero. "
        "For this experiment, perform the work yourself without delegation.")


In [ ]:
planning_agent = Agent(name='Planning 01', model=model,
    instructions=planning_system_prompt + "\n\n" + WRITE_TODOS_SYSTEM_PROMPT, tools=[write_todos, list_companies, read_company_financials, calculate_percentage], model_settings=BASE_SETTINGS)
print("Agent ready:", planning_agent.name)


In [ ]:
planning_report = begin_run("planning_001", planning_prompt, planning_agent.instructions,
    max_turns=40, source_recursion_limit=80, timeout_s=180)
planning_report["system_instruction_conversion"] = {
    "original_authored": planning_system_prompt, "new": planning_agent.instructions,
    "reason": "เพิ่ม TodoListMiddleware system text ให้ custom write_todos; ไม่มี built-in middleware ใน SDK นี้",
}
planning_report["configuration"]["planning_tool"] = "custom_function_tool_write_todos"
planning_context = {"todos": []}
planning_result = await run_once(planning_report, planning_agent, context=planning_context)
planning_report["final_todos"] = deepcopy(planning_context["todos"])
show_run(planning_report)


In [ ]:
def inspect_planning(record, data_names):
    calls = calls_of(record)
    plans = [c for c in calls if c["name"] == "write_todos" and isinstance(c["args"], dict)]
    data = [c for c in calls if c["name"] in data_names]
    checks = {
        "plan_created": bool(plans),
        "plan_before_data_calls": bool(plans and data) and plans[0]["response_index"] < data[0]["response_index"],
        "plan_changed_during_run": len({sha256_json(c["args"].get("todos", [])) for c in plans}) > 1,
        "calculator_used": any(c["name"] == "calculate_percentage" for c in calls),
        "final_answer_complete": final_complete(record),
    }
    record["plan_updates"] = deepcopy(plans)
    score(record, checks)
    return calls, plans

planning_calls, planning_updates = inspect_planning(planning_report, {"list_companies", "read_company_financials"})
add_review(planning_report, {
    "progress_matches_observed_work": "สถานะ To-do ตรงกับการทำงานที่เห็นใน trace",
    "all_metrics_and_sources_correct": "Delta 30/12/30, Echo 20/20/60, Foxtrot 20/N/A/20 และ source_id จาก fixtures ถูกต้อง",
    "missing_data_not_invented": "Foxtrot ขาดกำไรสุทธิ แสดง N/A; Delta/Echo มีข้อมูลครบ",
    "thai_table_and_summary": "คำตอบมีตารางเปรียบเทียบไทยและข้อสรุปตามโจทย์",
})
display(Markdown(planning_report["raw_answer"] or "*ไม่มีคำตอบ*"))
print(json.dumps(planning_report["plan_updates"], ensure_ascii=False, indent=2))
planning_001_review_path = save_review_template(planning_report)
print("Review template:", planning_001_review_path)
# apply_review(planning_report, planning_001_review_path) หลังตรวจหลักฐาน


### Planning02

In [ ]:
echo_backup_2025 = deepcopy(PLANNING_DATA["co-e42"])


In [ ]:
echo_backup_2025["source_id"] = "fixture_echo_backup_2025"

echo_backup_2024 = {
    **deepcopy(PLANNING_DATA["co-e42"]),
    "year": 2024,
    "previous_year": 2023,
    "revenue_previous": 700,
    "revenue_current": 800,
    "net_profit_current": 128,
    "liabilities": 880,
    "assets": 1500,
    "source_id": "fixture_echo_backup_2024",
}

BACKUP_SOURCES = {
    "fixture_echo_backup_2024": echo_backup_2024,
    "fixture_echo_backup_2025": echo_backup_2025,
}


In [ ]:
@function_tool(failure_error_function=None)
def read_primary_financials(company_id: str) -> dict:
    """Read annual financial data from the primary source."""
    if company_id == "co-e42":
        return {
            "company_id": company_id,
            "error": "PRIMARY_SOURCE_UNAVAILABLE",
            "retryable": False,
        }

    row = PLANNING_DATA.get(company_id)

    if row is None:
        return {"error": "COMPANY_NOT_FOUND"}

    return deepcopy(row)

@function_tool(failure_error_function=None)
def find_financial_sources(company_id: str) -> list[dict]:
    """Find alternative financial documents for a company.

    Check each document's reporting year before using it.
    """
    return [
        {
            "source_id": source_id,
            "company_id": row["company_id"],
            "reporting_year": row["year"],
            "document_type": "annual_financial_statement",
        }
        for source_id, row in BACKUP_SOURCES.items()
        if row["company_id"] == company_id
    ]

@function_tool(failure_error_function=None)
def read_financial_source(source_id: str) -> dict:
    """Read a financial document by its source ID."""
    row = BACKUP_SOURCES.get(source_id)

    if row is None:
        return {"error": "SOURCE_NOT_FOUND"}

    return deepcopy(row)


In [ ]:
planning_002_prompt = """
เปรียบเทียบบริษัททั้งหมดในระบบ โดยใช้ข้อมูลปี 2025
ในสามด้าน:
- การเติบโตของรายได้เทียบปีก่อน
- อัตรากำไรสุทธิ
- หนี้สินรวมต่อสินทรัพย์รวม

ระบุว่าบริษัทใดมีข้อมูลครบสำหรับคำนวณทั้งสามตัวชี้วัด
และบริษัทใดยังขาดข้อมูลอะไร

ข้อกำหนด:
- สร้างแผนผ่าน write_todos ก่อนเริ่มดึงข้อมูล
- เลือกงานย่อยและลำดับการทำงานเอง
- เริ่มตรวจข้อมูลจากแหล่งหลัก
- เมื่อพบอุปสรรค ให้ประเมินทางเลือกและอัปเดตแผนก่อนดำเนินทางเลือกนั้น
- ไม่เรียกซ้ำโดยไม่มีเหตุผลเมื่อ error ระบุ retryable เป็น false
- ตรวจบริษัทและปีของเอกสารให้ตรงกับงานก่อนนำตัวเลขมาใช้
- ใช้ calculate_percentage สำหรับการคำนวณ
- ข้อมูลขาดให้แสดง N/A และอธิบาย ห้ามสมมติตัวเลข
- อ้างอิง source_id ของข้อมูลที่ใช้จริง
- ตอบภาษาไทยเป็นตาราง พร้อมข้อสรุปสั้น ๆ
- อธิบายว่าพบอุปสรรคอะไรและแก้ไขอย่างไร
"""


In [ ]:
planning_002_system_prompt = ("Complete the analysis using the supplied financial tools. "
        "Maintain an actionable task list reflecting actual progress. "
        "Revise your plan when new evidence requires a different approach. "
        "Treat missing values as unavailable, not as zero. "
        "Perform the work yourself without delegation.")


In [ ]:
planning_002_agent = Agent(name='Planning 02', model=model,
    instructions=planning_002_system_prompt + "\n\n" + WRITE_TODOS_SYSTEM_PROMPT, tools=[write_todos, list_companies, read_primary_financials, find_financial_sources, read_financial_source, calculate_percentage], model_settings=BASE_SETTINGS)
print("Agent ready:", planning_002_agent.name)


In [ ]:
planning_002_report = begin_run("planning_002", planning_002_prompt, planning_002_agent.instructions,
    max_turns=40, source_recursion_limit=100, timeout_s=180)
planning_002_report["fixture_version"] = "planning_backup_v1"
planning_002_report["system_instruction_conversion"] = {
    "original_authored": planning_002_system_prompt, "new": planning_002_agent.instructions,
    "reason": "เพิ่ม TodoListMiddleware system text ให้ custom write_todos; ไม่มี built-in middleware ใน SDK นี้",
}
planning_002_report["configuration"]["planning_tool"] = "custom_function_tool_write_todos"
planning_002_context = {"todos": []}
planning_002_result = await run_once(planning_002_report, planning_002_agent, context=planning_002_context)
planning_002_report["final_todos"] = deepcopy(planning_002_context["todos"])
show_run(planning_002_report)


In [ ]:
calls_p2, plans_p2 = inspect_planning(planning_002_report, {"list_companies", "read_primary_financials"})
echo_primary = [c for c in calls_p2 if c["name"] == "read_primary_financials" and (c["args"] or {}).get("company_id") == "co-e42"]
alternatives = [c for c in calls_p2 if c["name"] == "find_financial_sources" and (c["args"] or {}).get("company_id") == "co-e42"]
error_result_indices = [i for i, event in enumerate(planning_002_report["tool_trace"])
    if event["kind"] == "tool_result" and event.get("call_id") in {c["id"] for c in echo_primary}
    and "PRIMARY_SOURCE_UNAVAILABLE" in json.dumps(event.get("output"), ensure_ascii=False)]
replanned_before_alternative = bool(alternatives and error_result_indices) and any(
    plan["trace_index"] > error_result_indices[0]
    and plan["response_index"] < alternatives[0]["response_index"] for plan in plans_p2)
score(planning_002_report, {**planning_002_report["checks"],
    "nonretryable_primary_not_repeated": len(echo_primary) == 1,
    "plan_updated_before_alternative_search": replanned_before_alternative,
    "correct_backup_company_and_year": any(c["name"] == "read_financial_source" and
        (c["args"] or {}).get("source_id") == "fixture_echo_backup_2025" for c in calls_p2),
    "all_plan_items_completed": bool(planning_002_report["final_todos"]) and all(
        t["status"] == "completed" for t in planning_002_report["final_todos"]),
})
add_review(planning_002_report, {
    "plan_content_adapted_to_obstacles": "แผนเปลี่ยนเนื้อหาเพื่อแก้อุปสรรคจริง ไม่ใช่เปลี่ยน status อย่างเดียว",
    "calculation_arguments_correct": "ตรวจ numerator/denominator/mode ทุก call จากข้อมูลปีและบริษัทที่ถูกต้อง",
    "final_metrics_correct": "Delta 30/12/30, Echo 20/20/60, Foxtrot 20/N/A/20",
    "missing_data_not_invented": "ไม่แทนกำไรสุทธิที่ขาดด้วยศูนย์หรือค่าปีเก่า",
    "source_ids_correct": "Echo ใช้ fixture_echo_backup_2025; Delta/Foxtrot ใช้ fixture เดิม",
    "thai_table_summary_and_obstacles": "ครบตารางไทย ข้อมูลขาด ข้อสรุป และอธิบายอุปสรรค/วิธีแก้",
})
display(Markdown(planning_002_report["raw_answer"] or "*ไม่มีคำตอบ*"))
print(json.dumps(planning_002_report["tool_trace"], ensure_ascii=False, indent=2))
planning_002_review_path = save_review_template(planning_002_report)
print("Review template:", planning_002_review_path)
# apply_review(planning_002_report, planning_002_review_path) หลังตรวจหลักฐาน


### บันทึกผลการทดสอบ planning

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
planning_export_path = export_results('planning', [planning_report, planning_002_report])
print("Saved:", planning_export_path)


# 6 Long context

ประเมินการจัดการ prompt หรือบริบทที่มีข้อมูลจำนวนมาก โดยตรวจความถูกต้องของข้อเท็จจริงที่ต้องดึงจากเนื้อหา และบันทึกผลเพื่อใช้เปรียบเทียบระหว่างกรณี

### Long context 01 (สรุปเอกสารยาว)

In [ ]:
LONG_CASE_ID = "long_context_001"

def make_background(start, stop):
    return "\n\n".join(
        (
            f"[บันทึกประกอบ B{i:03d}]\n"
            f"บริษัท Other-{i:03d} รายงานยอดขายปี 2025 "
            f"จำนวน {2000 + i * 17} ล้านบาท "
            f"และกำไรสุทธิ {100 + i} ล้านบาท "
            "ฝ่ายปฏิบัติการอยู่ระหว่างทบทวนแผนคลังสินค้า "
            "ข้อมูลนี้เป็นของบริษัทอื่น ไม่ใช่งบของบริษัทเป้าหมาย"
        )
        for i in range(start, stop)
    )

long_documents = "\n\n".join([
    """
[เอกสาร N1]
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1200 | กำไรสุทธิ 120

[เอกสาร O1]
บริษัท Orchid | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 900 | กำไรสุทธิ 90

[เอกสาร P1]
บริษัท Pine | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1500 | กำไรสุทธิ 150
""",
    make_background(1, 21),
    """
[ประกาศแก้ไข N2 | revision 2]
แทนที่เอกสาร N1 ทั้งฉบับ
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1250 | กำไรสุทธิ 125

[เอกสาร O-SEPARATE]
บริษัท Orchid | ปี 2025 | งบเฉพาะกิจการ | หน่วย ล้านบาท
รายได้ 700 | กำไรสุทธิ 140
""",
    make_background(21, 41),
    """
[ประกาศแก้ไข P2 | revision 2]
แทนที่เอกสาร P1 ทั้งฉบับ
บริษัท Pine | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1500 | กำไรสุทธิ: ไม่เปิดเผย
ห้ามนำกำไรสุทธิจากฉบับที่ถูกแทนที่มาเติม

[ประกาศแก้ไข N3 | revision 3]
แทนที่เอกสาร N2 ทั้งฉบับ
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1300 | กำไรสุทธิ 156
""",
    make_background(41, 61),
    """
[เอกสาร O-2024]
บริษัท Orchid | ปี 2024 | งบรวม | หน่วย ล้านบาท
รายได้ 800 | กำไรสุทธิ 160

[สำเนาประกาศเก่าสำหรับการตรวจสอบย้อนหลัง]
เอกสาร N2 | revision 2
บริษัท Northstar | ปี 2025 | งบรวม | หน่วย ล้านบาท
รายได้ 1250 | กำไรสุทธิ 125
สำเนานี้ไม่ใช่ revision ใหม่
""",
])

print("Dataset ready.")
print("Document characters:", f"{len(long_documents):,}")


In [ ]:
long_system_prompt = (
    "Read the supplied documents and follow the user's output schema. "
    "Use only information in the user message. "
    "Do not call tools, browse, delegate, or access files. "
    "Return only the requested JSON object."
)

long_prompt = """
อ่านเอกสารทั้งหมดแล้วสรุปเฉพาะ Northstar, Orchid และ Pine

ข้อกำหนด:
1. ใช้เฉพาะงบรวมปี 2025 หน่วยล้านบาท
2. เรียงบริษัทตามนี้เท่านั้น: Northstar, Orchid, Pine
3. หากมีประกาศแทนที่ ให้เลือก revision สูงสุดของงบที่ตรงเงื่อนไข
   ห้ามเลือกเพียงเพราะข้อความนั้นอยู่ท้ายเอกสาร
4. ประกาศแทนที่มีผลทั้งฉบับ ห้ามผสมข้อมูลกับฉบับเก่า
5. หากกำไรสุทธิไม่เปิดเผย ให้ใช้ null และ missing_fields=["net_profit_million"]
   หากเปิดเผย ให้ missing_fields=[]
6. source_id ต้องเป็นรหัสเอกสารที่ใช้จริง
7. ห้ามใช้ tools ทุกชนิด ใช้ข้อมูลในคำถามนี้เท่านั้น
8. ตอบ JSON object เท่านั้น ไม่ใส่ code fence หรือคำอธิบาย
9. ห้ามเพิ่ม keys นอกโครงสร้างที่กำหนด
10. ตัวเลขต้องเป็น JSON number ไม่ใช่ string

โครงสร้าง:
{
  "year": 2025,
  "unit": "million_thb",
  "companies": [
    {
      "company_name": "ชื่อบริษัท",
      "revenue_million": 0,
      "net_profit_million": null,
      "source_id": "รหัสเอกสาร",
      "missing_fields": []
    }
  ]
}

รายการ companies ต้องมีครบ 3 บริษัทตามลำดับที่กำหนด

เอกสาร:
""" + long_documents


In [ ]:
long_agent = Agent(name='Long Context', model=model,
    instructions=long_system_prompt, tools=[], model_settings=BASE_SETTINGS)
print("Agent ready:", long_agent.name)


In [ ]:
long_report = begin_run('long_context_001', long_prompt, long_agent.instructions,
    case_version=None, max_turns=10, source_recursion_limit=20, timeout_s=180)

long_report_result = await run_once(long_report, long_agent)
show_run(long_report)


In [ ]:
expected_long_rows = [
    {
        "company_name": "Northstar",
        "revenue_million": 1300,
        "net_profit_million": 156,
        "source_id": "N3",
        "missing_fields": [],
    },
    {
        "company_name": "Orchid",
        "revenue_million": 900,
        "net_profit_million": 90,
        "source_id": "O1",
        "missing_fields": [],
    },
    {
        "company_name": "Pine",
        "revenue_million": 1500,
        "net_profit_million": None,
        "source_id": "P2",
        "missing_fields": ["net_profit_million"],
    },
]


In [ ]:
def same_number(actual, expected):
    if expected is None:
        return actual is None
    return (
        type(actual) in (int, float)
        and math.isfinite(actual)
        and actual == expected
    )


In [ ]:
def score_long(record):
    parsed = record["answer"]
    obj = parsed if isinstance(parsed, dict) else {}
    rows = obj.get("companies", [])
    rows_valid = isinstance(rows, list) and len(rows) == 3 and all(isinstance(row, dict) for row in rows)
    def row_for(name):
        found = [r for r in (rows if rows_valid else []) if r.get("company_name") == name]
        return found[0] if len(found) == 1 else {}
    extraction = {expected["company_name"]: all(
        key in row_for(expected["company_name"])
        and same_number(row_for(expected["company_name"])[key], expected[key])
        for key in ("revenue_million", "net_profit_million")) for expected in expected_long_rows}
    revisions = {expected["company_name"]: (
        row_for(expected["company_name"]).get("source_id") == expected["source_id"]
        and row_for(expected["company_name"]).get("missing_fields") == expected["missing_fields"])
        for expected in expected_long_rows}
    constraints = {
        "exact_top_level_keys": isinstance(parsed, dict) and set(parsed) == {"year", "unit", "companies"},
        "correct_year_and_unit": type(obj.get("year")) is int and obj.get("year") == 2025 and obj.get("unit") == "million_thb",
        "company_count_and_order": rows_valid and [r.get("company_name") for r in rows] == [r["company_name"] for r in expected_long_rows],
        "exact_row_keys": rows_valid and all(set(r) == set(expected_long_rows[0]) for r in rows),
        "no_tool_calls": record["tool_call_count"] == 0,
        "final_answer_complete": final_complete(record),
    }
    axes = {"information_extraction": all(extraction.values()), "instruction_following": all(constraints.values()),
        "revision_and_missing_data": all(revisions.values()) and extraction["Northstar"] and extraction["Pine"]}
    record["axes"] = axes if record["runtime_status"] == "completed" else {k: None for k in axes}
    record["prompt_characters"] = len(record["prompt"])
    score(record, {**{"extraction:"+k: v for k,v in extraction.items()},
                   **{"revisions:"+k: v for k,v in revisions.items()}, **constraints})

long_report["fixture_version"] = "long_context_v1"
score_long(long_report)
show_run(long_report)


### Long context 002

In [ ]:
# แยกเอกสารเดิมตรงช่วงข้อมูลรบกวน
# เพื่อรักษาข้อมูลเป้าหมายและประกาศแก้ไขทุกฉบับไว้เหมือนเดิม
remaining = long_documents
document_parts = []

for start, stop in [(1, 21), (21, 41), (41, 61)]:
    old_background = make_background(start, stop)
    before, separator, remaining = remaining.partition(old_background)

    assert separator, f"ไม่พบข้อมูลรบกวนช่วง {start}–{stop - 1}"
    document_parts.append(before)

document_parts.append(remaining)

# เพิ่มจาก 60 เป็น 300 บันทึก โดยใช้หมายเลขไม่ซ้ำ
long_002_documents = (
    document_parts[0]
    + make_background(1, 101)
    + document_parts[1]
    + make_background(101, 201)
    + document_parts[2]
    + make_background(201, 301)
    + document_parts[3]
)

# ใช้คำสั่งเดิม เปลี่ยนเฉพาะเอกสาร
assert long_prompt.endswith(long_documents)

instruction_prefix = long_prompt[:-len(long_documents)]
long_002_prompt = instruction_prefix + long_002_documents

print("Original prompt characters:", f"{len(long_prompt):,}")
print("New prompt characters:", f"{len(long_002_prompt):,}")
print("Length ratio:", round(len(long_002_prompt) / len(long_prompt), 2))
print("Distractor records: 60 → 300")


In [ ]:
long_002_report = begin_run('long_context_002', long_002_prompt, long_agent.instructions,
    case_version=None, max_turns=10, source_recursion_limit=20, timeout_s=180)

long_002_report_result = await run_once(long_002_report, long_agent)
show_run(long_002_report)


In [ ]:
long_002_report["fixture_version"] = "long_context_v1_distractors_300"
long_002_report["configuration"].update({"distractor_records": 300, "fresh_conversation": True})
score_long(long_002_report)
show_run(long_002_report)
for record in [long_report, long_002_report]:
    print(record["case_id"], "characters=", record["prompt_characters"],
          "latency=", record["latency_s"], "usage=", record["usage"], "passed=", record["task_passed"])


### บันทึกผลการทดสอบ long_context

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
long_context_export_path = export_results('long_context', [long_report, long_002_report])
print("Saved:", long_context_export_path)


# 7 Session history

ทดสอบการเก็บและนำประวัติการสนทนากลับมาใช้ต่อเนื่อง รวมถึงการคงสถานะผ่านการเริ่ม kernel ใหม่ เพื่อดูว่า agent สามารถอ้างอิงบริบทก่อนหน้าได้ตามที่คาดหรือไม่

ใช้ `SQLiteSession(..., db_path=":memory:")` สำหรับ history_001/002 และ SQLite บน disk สำหรับ history_003 ซึ่งเป็น persistence ของ OpenAI SDK จริง

ส่งเข้า `Runner.run` เฉพาะ prompt ของ turn ใหม่ แล้ว SDK โหลดประวัติจาก session เอง ไม่ส่ง `to_input_list()` ซ้ำควบคู่ไปด้วย นับ usage เฉพาะ responses ใหม่ของเคสนั้น

### history_001 (สร้าง Session)

เริ่มบทสนทนาและเก็บ thread ID กับผลการตอบ เพื่อใช้เป็นบริบทสำหรับการสนทนาต่อเนื่อง

In [ ]:
HISTORY_CASE_ID = "history_001"

history_system_prompt = (
    "You are a session-memory test agent. "
    "Use only information from the current conversation thread. "
    "When asked for JSON, return JSON only without Markdown."
)


In [ ]:
history_agent = Agent(name='Session History', model=model,
    instructions=history_system_prompt, tools=[], model_settings=BASE_SETTINGS)
print("Agent ready:", history_agent.name)


In [ ]:
history_turn_1_prompt = """
จดจำข้อมูลต่อไปนี้ไว้สำหรับคำถามถัดไปใน session นี้:

บริษัท: Aurora
ปีงบประมาณ: 2025
รายได้รวม: 2400 ล้านบาท
กำไรสุทธิ: 288 ล้านบาท
อัตรากำไรสุทธิ: 12%

ตอบเพียงคำว่า CONTEXT_SAVED
"""


In [ ]:
history_turn_2_prompt = """
จากข้อมูลที่บันทึกไว้ก่อนหน้านี้ ตอบเป็น JSON object เท่านั้น:

{
  "company": "ชื่อบริษัท",
  "year": 0,
  "revenue_million": 0,
  "net_profit_million": 0,
  "net_margin_pct": 0
}

ห้ามเพิ่ม key อื่น และห้ามคำนวณหรือเดาตัวเลขใหม่
"""


In [ ]:
history_report = begin_run("history_001", history_turn_1_prompt, history_system_prompt,
    max_turns=10, source_recursion_limit=None, timeout_s=120)
history_report["configuration"]["source_recursion_note"] = "ต้นฉบับไม่กำหนด limit สำหรับ history_001"
history_report["fixture_version"] = "session_history_v1"
history_thread_id = f"history-{history_report['run_id']}"
history_session = SQLiteSession(history_thread_id)  # default :memory:
history_report["session_id"] = history_thread_id
history_turn_1 = await history_turn(history_report, history_agent, history_turn_1_prompt, history_session, "seed")
print(history_turn_1["raw_answer"])


In [ ]:
history_turns = [history_turn_1]
if history_turn_1["runtime_status"] == "completed":
    history_turn_2 = await history_turn(history_report, history_agent, history_turn_2_prompt, history_session, "recall")
    history_turns.append(history_turn_2)
combine_turns(history_report, history_turns)
history_items_after_001 = await history_session.get_items()
show_run(history_report)


In [ ]:
expected_history = {
    "company": "Aurora",
    "year": 2025,
    "revenue_million": 2400,
    "net_profit_million": 288,
    "net_margin_pct": 12,
}


In [ ]:
score(history_report, {
    "context_saved_turn_1": history_turn_1["raw_answer"] == "CONTEXT_SAVED",
    "same_thread_reused": len(history_turns) == 2 and all(t["session_id"] == history_thread_id for t in history_turns),
    "exact_json_structure": isinstance(history_report["answer"], dict) and set(history_report["answer"]) == set(expected_history),
    "remembered_values_correct": history_report["answer"] == expected_history,
    "no_tool_calls": history_report["tool_call_count"] == 0,
    "final_answer_complete": len(history_turns) == 2 and final_complete(history_report),
})
show_run(history_report)


### History_002 (ดึงข้อมูลจาก Thread เดิม)

ส่งข้อความต่อใน thread เดิมและตรวจว่าคำตอบอาศัยบริบทจาก turn ก่อนหน้าได้ถูกต้อง

In [ ]:
history_correction_prompt = """
แก้ไขข้อมูลของบริษัทที่เราคุยกันก่อนหน้านี้:

- กำไรสุทธิที่ถูกต้องคือ 360 ล้านบาท แทนค่าเดิม
- อัตรากำไรสุทธิที่ถูกต้องคือ 15% แทนค่าเดิม
- ชื่อบริษัท ปีงบประมาณ และรายได้รวม คงเดิม

จากนี้ให้ใช้ค่าที่แก้ไขแล้ว
ตอบเพียงคำว่า CONTEXT_UPDATED
"""

history_recall_prompt = """
สรุปข้อมูลล่าสุดของบริษัทที่เราคุยกันไว้
ใช้ค่าที่แก้ไขล่าสุด และคงข้อมูลที่ไม่ได้ถูกแก้ไข

ตอบ JSON object เท่านั้น มี keys ดังนี้:
company, year, revenue_million, net_profit_million, net_margin_pct

ห้ามเพิ่ม key อื่น ห้ามใช้ tools และห้ามเดาข้อมูล
"""


In [ ]:
history_002_report = begin_run("history_002", history_correction_prompt, history_system_prompt,
    max_turns=10, source_recursion_limit=20, timeout_s=120)
history_002_report.update({"depends_on_run_id": history_report["run_id"],
    "fixture_version": "session_history_correction_v1", "session_id": history_thread_id})
history_before_correction = await history_session.get_items()
history_002_turns = []
for turn_name, prompt in [("correction", history_correction_prompt), ("recall", history_recall_prompt)]:
    turn = await history_turn(history_002_report, history_agent, prompt, history_session, turn_name)
    history_002_turns.append(turn)
    print(turn_name, turn["raw_answer"])
    if turn["runtime_status"] != "completed":
        break
combine_turns(history_002_report, history_002_turns)
history_after_correction = await history_session.get_items()
show_run(history_002_report)


In [ ]:
expected_h2 = {
    "company": "Aurora",
    "year": 2025,
    "revenue_million": 2400,
    "net_profit_million": 360,
    "net_margin_pct": 15,
}


In [ ]:
object_h2 = history_002_report["answer"] if isinstance(history_002_report["answer"], dict) else {}
def history_number_matches(key):
    actual = object_h2.get(key)
    return type(actual) in (int, float) and math.isfinite(actual) and actual == expected_h2[key]
score(history_002_report, {
    "correction_acknowledged": bool(history_002_turns) and history_002_turns[0]["raw_answer"] == "CONTEXT_UPDATED",
    "original_messages_retained": bool(history_before_correction)
        and history_after_correction[:len(history_before_correction)] == history_before_correction,
    "exact_json_structure": isinstance(history_002_report["answer"], dict) and set(object_h2) == set(expected_h2),
    "unchanged_fields_retained": object_h2.get("company") == "Aurora" and type(object_h2.get("year")) is int
        and object_h2.get("year") == 2025 and history_number_matches("revenue_million"),
    "latest_profit_used": history_number_matches("net_profit_million"),
    "latest_margin_used": history_number_matches("net_margin_pct"),
    "no_tool_calls": history_002_report["tool_call_count"] == 0,
    "no_invalid_tool_calls": not any(c["args"] is None for c in calls_of(history_002_report)),
    "final_answer_complete": len(history_002_turns) == 2 and final_complete(history_002_report),
})
history_002_report["session_items_before"] = deepcopy(history_before_correction)
history_002_report["session_items_after"] = deepcopy(history_after_correction)
show_run(history_002_report)


### บันทึกผลการทดสอบ session_history

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
session_history_export_path = export_results('session_history', [history_report, history_002_report])
print("Saved:", session_history_export_path)


### history 003 (บันทึกข้อมูลใน SQLite)

บันทึกและเตรียมข้อมูลประวัติการสนทนาเพื่อใช้ตรวจการกู้คืนสถานะในขั้นตอนถัดไป

In [ ]:
restart_system_prompt = (
    "Use only information from the current conversation thread. "
    "Do not call tools or access files. "
    "Follow the requested response format exactly."
)

restart_seed_prompt = """
จดจำข้อมูลสำหรับการสนทนานี้:

บริษัท: Vega
ปีงบประมาณ: 2025
รายได้รวม: 3200 ล้านบาท
กำไรสุทธิ: 480 ล้านบาท
อัตรากำไรสุทธิ: 15%
รหัสอ้างอิง: VEGA-73-KP

ตอบเพียงคำว่า CONTEXT_SAVED
"""


In [ ]:
history_003_report = begin_run("history_003", restart_seed_prompt, restart_system_prompt,
    max_turns=10, source_recursion_limit=20, timeout_s=120)
restart_run_id = history_003_report["run_id"]
restart_dir = STATE_DIR / restart_run_id
restart_dir.mkdir(parents=True, exist_ok=False)
restart_db_path = restart_dir / "session.sqlite"
restart_manifest_path = restart_dir / "manifest.json"
restart_session = SQLiteSession(f"history-003-{restart_run_id}", db_path=restart_db_path)
restart_agent = Agent(name="Persistent Session", model=model,
    instructions=restart_system_prompt, model_settings=BASE_SETTINGS)
seed_turn = await history_turn(history_003_report, restart_agent, restart_seed_prompt, restart_session, "seed")
seed_items = await restart_session.get_items()
restart_session.close()
combine_turns(history_003_report, [seed_turn])
history_003_report["session_id"] = f"history-003-{restart_run_id}"
history_003_report["configuration"].update({"session_backend": "SQLiteSession", "restart_type": "jupyter_kernel_restart"})
seed_retained = any(item.get("role") == "user" and item.get("content") == restart_seed_prompt for item in seed_items)
history_003_report["seed_context_acknowledged"] = seed_turn["raw_answer"] == "CONTEXT_SAVED"
history_003_report["checks"] = {"seed_persisted": seed_retained, "restart_completed": None}
history_003_report["task_passed"] = None
history_003_report["phase"] = "awaiting_restart" if seed_turn["runtime_status"] == "completed" and seed_retained else "seed_failed"
history_003_report["revision"] = 0
manifest = {
    "sdk": "openai_agents", "case_id": "history_003", "run_id": restart_run_id,
    "model": MODEL, "versions": deepcopy(PACKAGE_VERSIONS),
    "session_id": history_003_report["session_id"], "db_filename": "session.sqlite",
    "system_prompt": restart_system_prompt, "pid_before": os.getpid(),
    "seed_items_sha256": sha256_json(seed_items), "seed_item_count": len(seed_items),
    "record": deepcopy(history_003_report),
}
with restart_manifest_path.open("x", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2, allow_nan=False)
restart_prepared_export = export_results("session_history", [history_003_report])
restart_probe = "this_variable_must_disappear"
print("Manifest:", restart_manifest_path)
print("Saved:", restart_prepared_export)
print("Phase:", history_003_report["phase"], "— ยังไม่ให้คะแนน restart")


### ⏸ Restart Kernel ตรงนี้

1. ตรวจว่า seed มี `phase=awaiting_restart` และ manifest/SQLite ถูกบันทึกแล้ว
2. เลือก **Kernel → Restart Kernel** (ไม่ใช่ Restart and Run All)
3. รันเฉพาะเซลล์ **Config (หัวข้อ 0)** เพื่อคืน imports/helpers/client; ไม่รัน Smoke หรือ seed อีก
4. ข้ามมารันเซลล์ **กู้ session หลัง Restart** ด้านล่าง

ถ้ามีหลาย manifest ที่ยังไม่ resume ให้ระบุ `RESUME_RUN_ID` จากรายการที่เซลล์แสดง ไม่มี path/run ID เก่าของ Deep Agents ถูกฝังไว้ การ resume run เดิมเรียกโมเดลได้ครั้งเดียว; ถ้าต้องทดลองใหม่ให้เริ่ม seed ใหม่เพื่อได้ run_id ใหม่

In [ ]:
# Barrier นี้ตั้งใจหยุด Run All เพื่อให้เกิด restart จริง
if "restart_probe" in globals():
    raise RuntimeError("กรุณา Restart Kernel แล้วรันเฉพาะ Config ก่อนข้ามมาที่กู้ session")
print("Previous kernel variable is gone.")


### กู้ session หลัง Restart — ตรวจ disk state ก่อนเรียกโมเดล

In [ ]:
RESUME_RUN_ID = None  # หากพบหลายรายการ คัดลอก run_id ของ OpenAI ที่ต้องการจากรายการด้านล่าง
candidates = []
for path in sorted(STATE_DIR.glob("*/manifest.json")):
    candidate = json.loads(path.read_text(encoding="utf-8"))
    if (candidate.get("sdk") == "openai_agents" and candidate.get("case_id") == "history_003"
        and candidate.get("record", {}).get("phase") == "awaiting_restart"
        and not (path.parent / "resume_started.json").exists()):
        candidates.append((path, candidate))
if RESUME_RUN_ID:
    candidates = [(p, m) for p, m in candidates if m["run_id"] == RESUME_RUN_ID]
if len(candidates) != 1:
    print("Available run IDs:", [m["run_id"] for _, m in candidates])
    raise RuntimeError("ต้องเลือก manifest เพียงหนึ่งรายการ; ตั้ง RESUME_RUN_ID แล้วรันเซลล์นี้ใหม่")
manifest_path, manifest = candidates[0]
print("Selected manifest:", manifest_path)
# resume prompt เหมือนต้นฉบับ ไม่มีการเติมตัวเลขคำตอบจาก manifest เข้า input


In [ ]:
resume_prompt = """
จากข้อมูลที่ให้ไว้ก่อนหน้านี้ในบทสนทนานี้
ตอบ JSON object เท่านั้น โดยมี keys:
company, year, revenue_million, net_profit_million,
net_margin_pct, reference_code

ใช้ข้อมูลและรหัสอ้างอิงจากประวัติเดิม
ห้ามเดา ห้ามเพิ่ม keys และห้ามใช้ tools
หากไม่พบข้อมูลของ field ใด ให้ใช้ null
"""


In [ ]:
history_003_report = deepcopy(manifest["record"])
history_003_report["supersedes_record_sha256"] = sha256_json(manifest["record"])
history_003_report["revision"] = manifest["record"].get("revision", 0) + 1
history_003_report["phase"] = "resumed"
pid_before, pid_after = manifest["pid_before"], os.getpid()
restored_items = []
resume_turn = None
restore_checks = {"process_changed": pid_after != pid_before and "restart_probe" not in globals(),
                  "versions_unchanged": PACKAGE_VERSIONS == manifest["versions"],
                  "model_unchanged": MODEL == manifest["model"],
                  "seed_messages_restored_before_call": False}
restore_started = time.perf_counter()
restored_session = None
try:
    if not all(restore_checks[k] for k in ("process_changed", "versions_unchanged", "model_unchanged")):
        raise RuntimeError("restart/process/package/model ไม่ตรงเงื่อนไข; ยังไม่เรียกโมเดล")
    db_path = manifest_path.parent / manifest["db_filename"]
    if not db_path.is_file():
        raise FileNotFoundError("ไม่พบ SQLite session บน disk")
    restored_session = SQLiteSession(manifest["session_id"], db_path=db_path)
    restored_items = await restored_session.get_items()
    restore_checks["seed_messages_restored_before_call"] = bool(restored_items) and (
        len(restored_items) == manifest["seed_item_count"] and sha256_json(restored_items) == manifest["seed_items_sha256"])
    if not restore_checks["seed_messages_restored_before_call"]:
        raise RuntimeError("ประวัติบน disk ไม่ตรงกับ seed ก่อน restart; ยังไม่เรียกโมเดล")
    print("Messages restored before model call:", len(restored_items))
    restore_setup_s = time.perf_counter() - restore_started
    # สร้าง marker แบบ exclusive ก่อน request เพื่อกัน rerun เดิมเงียบ ๆ
    with (manifest_path.parent / "resume_started.json").open("x", encoding="utf-8") as f:
        json.dump({"run_id": manifest["run_id"], "pid": pid_after, "started_at_utc": datetime.now(timezone.utc).isoformat()}, f)
    restored_agent = Agent(name="Persistent Session", model=model,
        instructions=manifest["system_prompt"], model_settings=BASE_SETTINGS)
    resume_turn = await history_turn(history_003_report, restored_agent, resume_prompt, restored_session, "resume")
    combine_turns(history_003_report, [*manifest["record"]["turns"], resume_turn])
    history_003_report["latency"]["setup_s"] = restore_setup_s
except Exception as exc:
    history_003_report["runtime_status"] = "failed"
    history_003_report["error"] = {"type": type(exc).__name__, "message": str(exc), "phase": "restore"}
    history_003_report["trace_complete"] = False
    history_003_report["raw_answer"] = None
    refresh_evidence(history_003_report)
finally:
    if restored_session is not None:
        restored_session.close()
history_003_report["restore_checks"] = restore_checks
history_003_report["restored_items_sha256"] = sha256_json(restored_items)
history_003_report["configuration"].update({"process_id_before": pid_before, "process_id_after": pid_after})
show_run(history_003_report)


In [ ]:
expected_resume = {
    "company": "Vega",
    "year": 2025,
    "revenue_million": 3200,
    "net_profit_million": 480,
    "net_margin_pct": 15,
    "reference_code": "VEGA-73-KP",
}


In [ ]:
actual_resume = history_003_report["answer"] if isinstance(history_003_report["answer"], dict) else {}
value_checks = {key: (
    type(actual_resume.get(key)) in (int, float) and math.isfinite(actual_resume[key]) and actual_resume[key] == expected
    if type(expected) is int else type(actual_resume.get(key)) is str and actual_resume[key] == expected)
    for key, expected in expected_resume.items()}
score(history_003_report, {**restore_checks,
    "exact_json_structure": isinstance(history_003_report["answer"], dict) and set(actual_resume) == set(expected_resume),
    "all_values_correct": all(value_checks.values()),
    "no_tool_calls": resume_turn is not None and resume_turn["tool_call_count"] == 0,
    "no_invalid_tool_calls": resume_turn is not None and not any(c["args"] is None for c in calls_of(resume_turn)),
    "final_answer_complete": resume_turn is not None and final_complete(resume_turn),
})
history_003_report["value_checks"] = value_checks if history_003_report["runtime_status"] == "completed" else {k: None for k in value_checks}
show_run(history_003_report)
restart_completed_export = export_results("session_history", [history_003_report])
print("Saved:", restart_completed_export)


# 8 Error handling

หัวข้อนี้ตรวจพฤติกรรมของ agent เมื่อเครื่องมือส่งข้อผิดพลาด ทั้งกรณีที่ไม่ควรลองซ้ำและกรณีที่ลองซ้ำได้แต่มีจำนวนครั้งจำกัด โดยตรวจคำขอ จำนวนครั้งที่เรียก และรูปแบบคำตอบสุดท้าย

### Error01 ข้อผิดพลาดที่ไม่อนุญาตให้ลองซ้ำ

กรณีแรกส่งข้อผิดพลาดแบบถาวรกลับจากเครื่องมือ และตรวจว่า agent หยุดเรียกซ้ำพร้อมรายงานสถานะและรหัสข้อผิดพลาดได้ถูกต้อง

In [ ]:
error_tool_log = []


In [ ]:
@function_tool(failure_error_function=None)
def fetch_restricted_statement(company_name: str, year: int) -> dict:
    """Request a company's annual financial statement."""
    result = {
        "error": {
            "code": "ACCESS_DENIED",
            "retryable": False,
            "message": "The current account cannot access this statement.",
        }
    }
    error_tool_log.append({
        "tool": "fetch_restricted_statement",
        "arguments": {
            "company_name": company_name,
            "year": year,
        },
        "output": deepcopy(result),
    })
    return result


In [ ]:
error_system_prompt = """
Use the supplied tool to retrieve financial statements.
If the tool returns an error, follow its retryable flag.
Do not invent financial data.
Do not delegate or use other tools.
Return only the JSON requested by the user.
"""

error_prompt = """
ขอดูงบปี 2025 ของบริษัท Aurora โดยเรียกเครื่องมือที่ให้

ข้อกำหนด:
- หากเครื่องมือคืน retryable=false ห้ามเรียกคำขอนั้นซ้ำ
- หากดึงข้อมูลไม่ได้ ให้รายงานข้อผิดพลาดตามจริง
- ห้ามใช้ตัวเลขจากความรู้เดิมหรือเดาตัวเลข
- ตอบ JSON object เท่านั้น มี keys ตามนี้:

{
  "status": "ok หรือ unavailable",
  "company_name": "Aurora",
  "year": 2025,
  "error_code": "รหัสข้อผิดพลาดจากเครื่องมือ หรือ null เมื่อสำเร็จ",
  "statement": "ข้อมูลงบจากเครื่องมือ หรือ null เมื่อไม่สำเร็จ"
}
"""


In [ ]:
error_agent = Agent(name='Non-retryable Error', model=model,
    instructions=error_system_prompt, tools=[fetch_restricted_statement], model_settings=BASE_SETTINGS)
print("Agent ready:", error_agent.name)


In [ ]:
error_001_report = begin_run('error_001', error_prompt, error_agent.instructions,
    case_version=None, max_turns=10, source_recursion_limit=12, timeout_s=120)
error_tool_log.clear()
error_001_report_result = await run_once(error_001_report, error_agent)
show_run(error_001_report)


In [ ]:
expected_error_answer = {
    "status": "unavailable",
    "company_name": "Aurora",
    "year": 2025,
    "error_code": "ACCESS_DENIED",
    "statement": None,
}


In [ ]:
error_trace = deepcopy(error_tool_log)
error_001_report["server_trace"] = error_trace
error_001_report["fixture_version"] = "nonretryable_error_v1"
error_001_report["failure_mode"] = "nonretryable_error_in_tool_result"
error_calls = calls_of(error_001_report)
score(error_001_report, {
    "tool_executed_once": len(error_trace) == 1,
    "correct_request": len(error_trace) == 1 and error_trace[0]["arguments"] == {"company_name": "Aurora", "year": 2025},
    "nonretryable_error_received": len(error_trace) == 1 and error_trace[0]["output"]["error"]["code"] == "ACCESS_DENIED"
        and error_trace[0]["output"]["error"]["retryable"] is False,
    "only_expected_tool_call": len(error_calls) == 1 and error_calls[0]["name"] == "fetch_restricted_statement",
    "exact_answer": isinstance(error_001_report["answer"], dict) and error_001_report["answer"] == expected_error_answer
        and type(error_001_report["answer"].get("year")) is int,
    "no_invalid_tool_calls": not any(c["args"] is None for c in error_calls),
    "final_answer_complete": final_complete(error_001_report),
})
show_run(error_001_report)


### Error02 จำกัดจำนวนครั้งเมื่อลองใหม่

กรณีที่สองส่งข้อผิดพลาดชั่วคราวและกำหนดให้ลองซ้ำได้หนึ่งครั้ง จึงตรวจว่า agent เรียกเครื่องมือรวมไม่เกินสองครั้งและแจ้งจำนวนครั้งจริงในคำตอบ

In [ ]:
retry_limit_log = []


In [ ]:
@function_tool(failure_error_function=None)
def fetch_temporarily_unavailable_statement(
    company_name: str,
    year: int,
) -> dict:
    """Request an annual financial statement from the data service."""
    result = {
        "error": {
            "code": "TEMPORARY_UNAVAILABLE",
            "retryable": True,
            "message": "The financial data service is temporarily unavailable.",
        }
    }

    retry_limit_log.append({
        "tool": "fetch_temporarily_unavailable_statement",
        "arguments": {
            "company_name": company_name,
            "year": year,
        },
        "output": deepcopy(result),
    })

    return result


In [ ]:
retry_limit_system_prompt = """
Use the supplied tool to retrieve financial statements.
Follow the retry limit specified by the user.
Do not invent financial data.
Do not delegate or use other tools.
Return only the requested JSON.
"""

retry_limit_prompt = """
ขอดูงบปี 2025 ของบริษัท Aurora โดยเรียกเครื่องมือที่ให้

นโยบายจัดการข้อผิดพลาด:
- หากเครื่องมือคืน retryable=true ให้ลองคำขอเดิมซ้ำอีก 1 ครั้ง
- อนุญาตให้เรียกเครื่องมือรวมสูงสุด 2 ครั้ง
- หากครั้งที่สองยังผิดพลาด ต้องหยุดและรายงานว่าไม่สามารถดึงข้อมูลได้
- ห้ามเดาตัวเลขหรือใช้ข้อมูลจากความรู้เดิม

ตอบ JSON object เท่านั้น มี keys:
{
  "status": "ok หรือ unavailable",
  "company_name": "Aurora",
  "year": 2025,
  "error_code": "รหัสข้อผิดพลาดล่าสุด หรือ null เมื่อสำเร็จ",
  "attempts": 0,
  "statement": "ข้อมูลงบ หรือ null เมื่อไม่สำเร็จ"
}

attempts ต้องเป็นจำนวนครั้งที่เรียกเครื่องมือจริง
"""


In [ ]:
retry_limit_agent = Agent(name='Retry Exhaustion', model=model,
    instructions=retry_limit_system_prompt, tools=[fetch_temporarily_unavailable_statement], model_settings=BASE_SETTINGS)
print("Agent ready:", retry_limit_agent.name)


In [ ]:
error_002_report = begin_run('error_002', retry_limit_prompt, retry_limit_agent.instructions,
    case_version=None, max_turns=10, source_recursion_limit=16, timeout_s=120)
retry_limit_log.clear()
error_002_report_result = await run_once(error_002_report, retry_limit_agent)
show_run(error_002_report)


In [ ]:
expected_request = {
    "company_name": "Aurora",
    "year": 2025,
}

expected_retry_answer = {
    "status": "unavailable",
    "company_name": "Aurora",
    "year": 2025,
    "error_code": "TEMPORARY_UNAVAILABLE",
    "attempts": 2,
    "statement": None,
}


In [ ]:
retry_limit_trace = deepcopy(retry_limit_log)
error_002_report["server_trace"] = retry_limit_trace
error_002_report["fixture_version"] = "retry_exhaustion_v1"
error_002_report["failure_mode"] = "retryable_error_in_tool_result"
error_002_report["configuration"]["max_tool_attempts_by_prompt"] = 2
retry_calls = calls_of(error_002_report)
retry_after_error = len(retry_calls) == 2 and any(
    event["kind"] == "tool_result" and event.get("call_id") == retry_calls[0]["id"]
    for event in error_002_report["tool_trace"][retry_calls[0]["trace_index"]+1:retry_calls[1]["trace_index"]])
retry_answer = error_002_report["answer"]
score(error_002_report, {
    "exactly_two_executions": len(retry_limit_trace) == 2,
    "same_correct_request": len(retry_limit_trace) == 2 and all(e["arguments"] == expected_request for e in retry_limit_trace),
    "both_attempts_failed": len(retry_limit_trace) == 2 and all(
        e["output"]["error"]["code"] == "TEMPORARY_UNAVAILABLE" and e["output"]["error"]["retryable"] is True for e in retry_limit_trace),
    "only_expected_tool_calls": len(retry_calls) == 2 and all(c["name"] == "fetch_temporarily_unavailable_statement" for c in retry_calls),
    "retry_after_first_error": retry_after_error,
    "exact_answer": isinstance(retry_answer, dict) and retry_answer == expected_retry_answer
        and type(retry_answer.get("year")) is int and type(retry_answer.get("attempts")) is int,
    "no_invalid_tool_calls": not any(c["args"] is None for c in retry_calls),
    "final_answer_complete": final_complete(error_002_report),
})
show_run(error_002_report)


### บันทึกผลการทดสอบ error_handling

เก็บทุกสถานะโดยไม่บังคับให้ผ่านก่อนบันทึก การ export ซ้ำคง `run_id` เดิม และสร้างไฟล์ใหม่เสมอ

In [ ]:
error_handling_export_path = export_results('error_handling', [error_001_report, error_002_report])
print("Saved:", error_handling_export_path)


# 9 สรุปและสำรวจผล Benchmark

อ่านไฟล์ผลการทดลองที่บันทึกไว้ รวมรายการซ้ำและปัญหาการอ่านไฟล์ แล้วแสดงสถานะของแต่ละเคสในตารางเพื่อให้ตรวจภาพรวมได้สะดวก

อ่านเฉพาะ `shared/results/openai_agents` และกรอง `sdk` อีกชั้น แยก `(case_id, case_version, fixture_version, model)` ในตารางรวม; Smoke แยกจาก 17 เคสหลัก

นับแต่ละ `run_id` เพียงครั้งเดียวแม้ export ซ้ำ ถ้า review หรือ restart ทำให้ผลเดิมเปลี่ยน ต้องมี revision และ hash อ้างกลับ snapshot ก่อนหน้า; ถ้าเนื้อหาขัดแย้งโดยไม่มีสายอ้างอิง จะรายงาน `export_conflict` และไม่นำคะแนนใดมาเลือกเอง

`runtime_failed` แยกจาก `failed` (คำตอบ/พฤติกรรมไม่ผ่าน), `pending_review` และ `awaiting_restart` ไม่มีข้อความวิเคราะห์หรือคะแนนจาก Deep Agents ปะปน

In [ ]:
MAIN_CASE_IDS = [
    "math_001", "math_002", "math_003", "mcp_001", "mcp_002", "mcp_003",
    "web_native_001", "web_native_002", "planning_001", "planning_002",
    "long_context_001", "long_context_002", "history_001", "history_002", "history_003",
    "error_001", "error_002",
]

def collect_saved_runs(results_dir):
    grouped = defaultdict(list)
    issues = []
    for path in sorted(results_dir.glob("*.json")):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
            if payload.get("sdk") != "openai_agents":
                continue
            if not isinstance(payload.get("results"), list):
                raise ValueError("results ไม่ใช่ list")
            for record in payload["results"]:
                if not isinstance(record, dict):
                    raise ValueError("result ไม่ใช่ object")
                if record.get("sdk") != "openai_agents":
                    continue
                if not record.get("run_id") or not record.get("case_id"):
                    issues.append({"file": path.name, "error": "ไม่มี run_id/case_id; ไม่เดาว่าเป็น run ใหม่"})
                    continue
                key = (record["sdk"], record["case_id"], record.get("case_version"), record["run_id"])
                grouped[key].append({"record": record, "file": path.name})
        except (OSError, ValueError, AttributeError) as exc:
            issues.append({"file": path.name, "error": str(exc)})
    saved = []
    duplicate_exports = 0
    for key, entries in grouped.items():
        duplicate_exports += len(entries) - 1
        unique = {sha256_json(e["record"]): e for e in entries}
        ordered = sorted(unique.items(), key=lambda pair: pair[1]["record"].get("revision", 0))
        selected_hash, selected = ordered[0]
        conflict = False
        for item_hash, item in ordered[1:]:
            if (item["record"].get("revision", 0) == selected["record"].get("revision", 0) + 1
                and item["record"].get("supersedes_record_sha256") == selected_hash):
                selected_hash, selected = item_hash, item
            else:
                conflict = True
        selected = deepcopy(selected)
        selected["export_conflict"] = conflict
        selected["export_files"] = [e["file"] for e in entries]
        if conflict:
            issues.append({"run_id": key[-1], "error": "conflicting exports without a verified revision chain"})
        saved.append(selected)
    return saved, duplicate_exports, issues

def run_status(item):
    record = item["record"]
    if item.get("export_conflict"):
        return "export_conflict"
    if record["runtime_status"] != "completed":
        return "runtime_failed"
    if record.get("phase") == "awaiting_restart":
        return "awaiting_restart"
    return "passed" if record.get("task_passed") is True else "failed" if record.get("task_passed") is False else "pending_review"

saved_runs, duplicate_exports, load_issues = collect_saved_runs(RESULTS_DIR)
rows = []
for item in sorted(saved_runs, key=lambda e: (e["record"]["case_id"], str(e["record"].get("case_version")), e["record"]["run_id"])):
    record = item["record"]
    rows.append({"case_id": record["case_id"], "case_version": record.get("case_version"),
        "fixture_version": record.get("fixture_version"), "run_id": record["run_id"],
        "model": record["model"], "status": run_status(item),
        "task_passed": None if item["export_conflict"] else record.get("task_passed"),
        "source_file": item["file"], "export_files": item["export_files"]})
table = ["| Case | Version | Run | Model | Status |", "|---|---|---|---|---|"]
for row in rows:
    table.append(f"| {row['case_id']} | {row['case_version'] or 'ไม่ได้ระบุในต้นฉบับ'} | {row['run_id'][:8]} | {row['model']} | {row['status']} |")
display(Markdown("\n".join(table)))
print("Unique saved runs:", len(rows), "| duplicate exports:", duplicate_exports)
print("Missing main cases:", sorted(set(MAIN_CASE_IDS) - {r["case_id"] for r in rows}))
print("Load issues:", json.dumps(load_issues, ensure_ascii=False, indent=2))


### สร้างสรุปผลสำหรับนำไปใช้ต่อ

เซลล์สุดท้ายจัดกลุ่มผลตามเคส นับสถานะ และบันทึกสรุปพร้อมหมายเหตุเกี่ยวกับข้อจำกัดของการทดลองเป็นไฟล์ JSON

In [ ]:
group_counts = defaultdict(Counter)
for row in rows:
    key = (row["case_id"], row["case_version"], row["fixture_version"], row["model"])
    group_counts[key][row["status"]] += 1
main_rows = [r for r in rows if r["case_id"] in MAIN_CASE_IDS]
summary = {
    "schema_version": "1.0", "saved_at_utc": datetime.now(timezone.utc).isoformat(),
    "sdk": "openai_agents", "category": "benchmark_summary", "results": rows,
    "expected_main_case_count": 17, "main_run_count": len(main_rows),
    "main_status_counts": dict(Counter(r["status"] for r in main_rows)),
    "smoke_status_counts": dict(Counter(r["status"] for r in rows if r["case_id"] == "smoke_001")),
    "duplicate_exports": duplicate_exports, "load_issues": load_issues,
    "groups": [{"case_id": k[0], "case_version": k[1], "fixture_version": k[2], "model": k[3], "status_counts": dict(v)}
               for k, v in group_counts.items()],
    "limitations": [
        "ผลสะท้อน model, SDK version, system scaffolding และ fixtures ที่บันทึกไว้",
        "ไม่มีคะแนนหรือคำวิเคราะห์รอบเก่าของ Deep Agents ถูกนำมาใช้",
        "recursion_limit กับ max_turns ไม่ใช่หน่วยเดียวกัน",
        "MCP transport เปลี่ยนจาก in-memory เป็น localhost Streamable HTTP",
        "Planning ใช้ custom write_todos; prompts ขอให้วางแผนอย่างชัดเจน",
        "Web open_page action และ citations อย่างเดียวไม่ยืนยัน content support",
        "usage unavailable เป็น null; partial usage ไม่ใช้เป็นยอดรวมที่ครบ",
        "ผลรอบเดียวไม่บอกการกระจาย latency หรือความเสถียร และยังเลือกผู้ชนะข้าม SDK ไม่ได้",
    ],
}
summary_path = SUMMARY_DIR / f"openai_agents_{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}_{uuid4().hex}.json"
with summary_path.open("x", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, allow_nan=False)
print("Saved summary:", summary_path)
